# Multilingual YouTube Analyzer

Final Project — Ironhack AI Bootcamp

**Final Project – Ironhack AI Bootcamp**

A multilingual RAG-based chatbot that transcribes YouTube videos via audio download + Whisper into a vector database and answers questions about the content using memory and a LangChain agent with 7 tools.

## Features

- 🎙️ **Audio-based ingestion** (yt-dlp → Whisper)
- 🔄 **Switchable Whisper backends**: OpenAI API or open-source Whisper (tiny/base/small/medium/large-v3)
- 🛠️ **7 agent tools**: Search, List, Add, Metadata, Summarize, Compare, Key Moments
- 🎤 **Audio input** via microphone for user questions
- ✅ **Ground-truth check**: WER comparison Whisper vs. YouTube captions
- ⚖️ **LLM-as-Judge faithfulness eval** for agent answers
- 📊 **LangSmith tracing** (EU endpoint)
- 🌍 **Multilingual**: responds in the language of the question
- 💾 **Persistent ChromaDB** (optional via Google Drive)

## Requirements

- **`OPENAI_API_KEY`**
- **`HF_TOKEN`** (for Llama / Qwen via HuggingFace Inference API)
- **`LANGSMITH_API_KEY`** (EU server, optional)


## 1. Installation

Installs all required libraries.

In [1]:
from pathlib import Path

_req_path = Path("/content/requirements.txt") if Path("/content").exists() else Path("requirements.txt")
_req_path.write_text("""langchain>=0.3
langchain-core>=0.3
langchain-openai
langchain-chroma
langchain-text-splitters
langchain-community
langgraph
langsmith
chromadb
openai
openai-whisper
tiktoken
yt-dlp
youtube-transcript-api
gradio
pydantic
tenacity
python-dotenv
jiwer
huggingface-hub
langchain-huggingface
sentence-transformers
requests
pandas
matplotlib
seaborn
numpy
torch
""")
print(f"✅ requirements.txt written to {_req_path}")


✅ requirements.txt written to /content/requirements.txt


In [2]:
import subprocess, sys

PACKAGES = [
    "langchain>=0.3",
    "langchain-core>=0.3",
    "langchain-openai",
    "langchain-chroma",
    "langchain-text-splitters",
    "langchain-community",
    "langgraph",
    "langsmith",
    "chromadb",
    "openai",
    "tiktoken",
    "yt-dlp",
    "youtube-transcript-api",
    "gradio",
    "pydantic",
    "tenacity",
    "python-dotenv",
    "openai-whisper",
    "jiwer",
    "huggingface-hub",
    "langchain-huggingface",
    "sentence-transformers",
    "requests",
]

print(f"📦 Installing {len(PACKAGES)} packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES, check=True)
print("✅ pip install finished")

print("\n🔍 Verifying imports...")
for pkg in ["yt_dlp", "whisper", "gradio", "langchain", "langgraph",
            "langchain_chroma", "langchain_openai", "openai", "tiktoken", "jiwer",
            "huggingface_hub", "langchain_huggingface", "requests"]:
    try:
        __import__(pkg)
        print(f"   ✅ {pkg}")
    except ImportError as e:
        print(f"   ❌ {pkg} – {e}")

print("\n✅ All packages installed and importable")


📦 Installing 23 packages...
✅ pip install finished

🔍 Verifying imports...
   ✅ yt_dlp
   ✅ whisper
   ✅ gradio
   ✅ langchain
   ✅ langgraph
   ✅ langchain_chroma
   ✅ langchain_openai
   ✅ openai
   ✅ tiktoken
   ✅ jiwer
   ✅ huggingface_hub
   ✅ langchain_huggingface
   ✅ requests

✅ All packages installed and importable


## 2. ffmpeg

Required by yt-dlp for audio processing.

In [3]:
import shutil, subprocess, sys

if shutil.which("ffmpeg") is None:
    if sys.platform == "linux":
        subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"], check=True)
    elif sys.platform == "darwin":
        subprocess.run(["brew", "install", "ffmpeg"], check=True)
    else:
        raise EnvironmentError("ffmpeg not found. Please install it manually: https://ffmpeg.org/download.html")
print("✅ ffmpeg:", shutil.which("ffmpeg"))


✅ ffmpeg: /usr/bin/ffmpeg


## 3. API Keys

Priority order: `.env` file → `os.environ` → Colab Secrets. All values are cleaned with `.strip()`.


In [4]:
import os
import warnings
warnings.filterwarnings("ignore")

def load_secret(key_name: str, required: bool = True):
    """Load a secret: .env file -> os.environ -> Colab Secrets (in that order)."""
    value = None

    # 1. .env file (local development)
    if not value:
        try:
            from dotenv import load_dotenv
            load_dotenv(override=False)
            raw = os.environ.get(key_name, "")
            value = raw.strip() if raw else None
        except ImportError:
            pass

    # 2. Environment variable
    if not value:
        raw = os.environ.get(key_name, "")
        value = raw.strip() if raw else None

    # 3. Colab Secrets (only if running in Colab and still no value)
    if not value:
        try:
            from google.colab import userdata
            raw = userdata.get(key_name)
            if raw:
                value = raw.strip()
        except Exception:
            pass

    if value:
        if any(c in value for c in "\r\n\t"):
            raise ValueError(f"{key_name} contains invalid characters")
        os.environ[key_name] = value
        return value
    if required:
        raise RuntimeError(f"❌ {key_name} not found in .env, environment, or Colab Secrets.")
    return None


OPENAI_API_KEY = load_secret("OPENAI_API_KEY", required=True)
print(f"✅ OPENAI_API_KEY loaded (ends with: ...{OPENAI_API_KEY[-4:]})")

HF_TOKEN = load_secret("HF_TOKEN", required=False)
if HF_TOKEN:
    print(f"✅ HF_TOKEN loaded (ends with: ...{HF_TOKEN[-4:]})")
else:
    print("ℹ️  HF_TOKEN not set — HuggingFace models will be skipped")

LANGSMITH_API_KEY = load_secret("LANGSMITH_API_KEY", required=False)
if LANGSMITH_API_KEY:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "youtube-qa-bot"
    os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
    print("✅ LangSmith tracing active (EU)")


✅ OPENAI_API_KEY loaded (ends with: ...WhMA)
✅ HF_TOKEN loaded (ends with: ...UTvP)
✅ LangSmith tracing active (EU)


In [5]:
from pathlib import Path
import shutil

_is_colab = Path("/content").exists()
benchmark_path = Path("/content/benchmark_results") if _is_colab else Path("benchmark_results")
benchmark_path.mkdir(parents=True, exist_ok=True)

# Auto-copy benchmark exports placed directly in /content/
_expected = ["prices.json", "winners.json", "summary.json",
             "chunk_recommendations.json", "combo_table.csv"]
for _f in _expected:
    _src = Path("/content") / _f
    _dst = benchmark_path / _f
    if _src.exists() and not _dst.exists():
        shutil.copy(_src, _dst)
        print(f"   📋 Copied {_f} → benchmark_results/")

print(f"✅ Benchmark results directory: {benchmark_path}")


   📋 Copied prices.json → benchmark_results/
   📋 Copied winners.json → benchmark_results/
   📋 Copied summary.json → benchmark_results/
   📋 Copied chunk_recommendations.json → benchmark_results/
   📋 Copied combo_table.csv → benchmark_results/
✅ Benchmark results directory: /content/benchmark_results


## 4. Auto-load Benchmark Imports

Loads all PNGs, JSONs and CSVs that the Q&A notebook needs from
`benchmark_imports_for_QandA_notebook/`. Works in Colab AND locally:

- **Colab:** if the folder is missing, downloads everything from GitHub once.
- **Local:** uses the folder that lives next to this notebook.

After this runs, all benchmark plots and pre-computed data are ready for the
Gradio UI.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4. Auto-load Benchmark Imports
# ═══════════════════════════════════════════════════════════════════
# Loads everything the Q&A notebook needs (PNG plots, JSON tables, CSV stats)
# in one go. Works the same in Colab and locally:
#   • Local:  expected next to this notebook in benchmark_imports_for_QandA_notebook/
#   • Colab:  if missing, fetched automatically from your GitHub repo

from pathlib import Path
import os
import urllib.request

# Detect environment (same logic as the env-detection cell below)
def _is_colab():
    if not Path("/content").exists():
        return False
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

_IS_COLAB_LOADER = _is_colab()
_BASE_LOADER = Path("/content") if _IS_COLAB_LOADER else Path.cwd()
IMPORTS_DIR = _BASE_LOADER / "benchmark_imports_for_QandA_notebook"

# ───────────────────────────────────────────────────────────────
# CONFIGURE THIS to point to your repo if running in Colab
# ───────────────────────────────────────────────────────────────
GITHUB_USER = "DanielHuette"
GITHUB_REPO = "Multilingual_YouTube_Analyzer"
GITHUB_BRANCH = "main"
GITHUB_RAW_BASE = (
    f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/"
    f"{GITHUB_BRANCH}/benchmark_imports_for_QandA_notebook"
)

# Files the Q&A notebook expects to find
EXPECTED_FILES = [
    # Benchmark plots referenced by the Gradio UI
    "benchmark1.png",
    "benchmark2.png",
    "benchmark3.png",
    "benchmark4.png",
    "benchmark5.png",
    "benchmark6_short_video.png",
    "benchmark6_medium_video.png",
    "benchmark6_long_video.png",
    "benchmark6_short_video_curves.png",
    "benchmark6_medium_video_curves.png",
    "benchmark6_long_video_curves.png",
    "benchmark6_short_video_heatmap.png",
    "benchmark6_medium_video_heatmap.png",
    "benchmark6_long_video_heatmap.png",
    "benchmark7_full_matrix.png",
    # Pre-computed benchmark data (read by load_benchmark_data)
    "prices.json",
    "combo_table.csv",
    "winners.json",
    "chunk_recommendations.json",
    "summary.json",
    "benchmark1_whisper_sizes.csv",
    "benchmark2_bitrates.csv",
    "benchmark3_embeddings.csv",
    "benchmark4_llms.csv",
    "benchmark5_rag_vs_norag.csv",
    "benchmark6_chunk_grid.csv",
    "benchmark7_full_matrix.csv",
]


def _download_one(name: str, dest: Path) -> bool:
    """Download a single file from the configured GitHub raw URL."""
    url = f"{GITHUB_RAW_BASE}/{name}"
    try:
        urllib.request.urlretrieve(url, str(dest))
        return True
    except Exception as e:
        print(f"   ⚠️  {name}: {e}")
        return False


def ensure_benchmark_imports():
    """Make sure benchmark imports folder exists and has the expected files."""
    IMPORTS_DIR.mkdir(parents=True, exist_ok=True)

    # Inventory: what's there, what's missing
    present = {p.name for p in IMPORTS_DIR.iterdir() if p.is_file()}
    missing = [n for n in EXPECTED_FILES if n not in present]

    if not missing:
        print(f"✅ All {len(EXPECTED_FILES)} benchmark imports present "
              f"in {IMPORTS_DIR}")
        return

    print(f"📥 {len(missing)} of {len(EXPECTED_FILES)} files missing in "
          f"{IMPORTS_DIR}")

    if _IS_COLAB_LOADER:
        # In Colab → try to download from GitHub
        print(f"   Downloading from {GITHUB_RAW_BASE} …")
        ok = 0
        fail = 0
        for name in missing:
            if _download_one(name, IMPORTS_DIR / name):
                ok += 1
            else:
                fail += 1
        print(f"   Done: {ok} downloaded, {fail} failed.")
        if fail > 0:
            print("   ⚠️  Some files could not be downloaded. The notebook "
                  "still runs, but related UI tabs may show placeholders.")
    else:
        # Local → don't auto-download; tell the user what to do
        print("   Local mode: please make sure the folder")
        print(f"     {IMPORTS_DIR}")
        print("   exists next to this notebook and contains the listed "
              "files (re-run the benchmark notebook to regenerate them).")
        print("\n   Missing:")
        for name in missing[:10]:
            print(f"     • {name}")
        if len(missing) > 10:
            print(f"     • … and {len(missing) - 10} more")


ensure_benchmark_imports()


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4b. Environment detection — Colab vs. local
# ═══════════════════════════════════════════════════════════════════
# Stellt sicher, dass das Notebook unverändert sowohl in Google Colab als
# auch lokal läuft. Alle weiteren Zellen verwenden BASE_DIR als Wurzel
# für Pfade — und finden ihre Daten dadurch automatisch am richtigen Ort.

from pathlib import Path

# Colab erkennt man am `/content/`-Ordner und am google.colab-Modul.
def _detect_colab() -> bool:
    if not Path("/content").exists():
        return False
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IS_COLAB = _detect_colab()
BASE_DIR = Path("/content") if IS_COLAB else Path.cwd()

print(f"🌍 Environment: {'Colab' if IS_COLAB else 'Local'}")
print(f"📁 BASE_DIR:    {BASE_DIR}")


In [ ]:
CONFIG = {
    # LLM — gpt-4o-mini is the benchmark sweet-spot (B4: best quality/cost ratio)
    "agent_model":   "gpt-4o-mini",
    "embedding_model": "text-embedding-3-small",
    "helper_model":  "gpt-4o-mini",

    "available_agent_models": [
        "gpt-4o-mini",
        "gpt-5-mini",
        "gpt-5",
        "gpt-4o",
        "hf:llama-3.3",
        "hf:qwen2.5-72b",
    ],

    "hf_model_map": {
        "hf:llama-3.3":   "meta-llama/Llama-3.3-70B-Instruct",
        "hf:qwen2.5-72b": "Qwen/Qwen2.5-72B-Instruct",
    },

    # Whisper — api (whisper-1) is the B1 benchmark winner
    "available_whisper_options": [
        "api (whisper-1)",
        "local: small",
        "local: base",
        "local: medium",
        "local: large-v3",
        "local: tiny",
    ],

    "whisper_backend":    "api",
    "whisper_local_size": "small",
    "audio_format":       "m4a",
    "audio_bitrate_kbps": "64",
    "whisper_max_mb":     24,

    # Chunk size is set dynamically at ingest time based on video duration.
    # These are neutral fallbacks only.
    "chunk_size":    500,
    "chunk_overlap": 100,
    "retrieval_k":   5,

    # Paths: BASE_DIR is /content/ in Colab, current dir locally.
    "use_drive":        False,
    "chroma_dir_local": str(BASE_DIR / "chroma_db"),
    "chroma_dir_drive": "/content/drive/MyDrive/youtube_qa_bot/chroma_db",
    "collection_name":  "youtube_videos",
    "audio_dir":        str(BASE_DIR / "audio_cache"),
    "keep_audio_files": False,

    # Benchmark imports folder (PNGs/JSONs/CSVs from benchmark.ipynb)
    "benchmark_imports_dir": str(BASE_DIR / "benchmark_imports_for_QandA_notebook"),
}

print(f"✅ CONFIG ready")
print(f"   Agent:           {CONFIG['agent_model']}")
print(f"   Whisper:         {CONFIG['whisper_backend']}")
print(f"   Embedding:       {CONFIG['embedding_model']}")
print(f"   ChromaDB:        {CONFIG['chroma_dir_local']}")
print(f"   Audio cache:     {CONFIG['audio_dir']}")
print(f"   Benchmark imports: {CONFIG['benchmark_imports_dir']}")

In [7]:
if CONFIG["use_drive"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CONFIG["chroma_dir"] = CONFIG["chroma_dir_drive"]
        os.makedirs(CONFIG["chroma_dir"], exist_ok=True)
        print(f"✅ ChromaDB in Drive: {CONFIG['chroma_dir']}")
    except ImportError:
        CONFIG["chroma_dir"] = CONFIG["chroma_dir_local"]
        os.makedirs(CONFIG["chroma_dir"], exist_ok=True)
        print(f"ℹ️  Not in Colab — using local path: {CONFIG['chroma_dir']}")
else:
    CONFIG["chroma_dir"] = CONFIG["chroma_dir_local"]
    os.makedirs(CONFIG["chroma_dir"], exist_ok=True)
    print(f"ℹ️  ChromaDB local: {CONFIG['chroma_dir']}")

os.makedirs(CONFIG["audio_dir"], exist_ok=True)


ℹ️  ChromaDB local: /content/chroma_db


## 5. Benchmark Data

Loads `prices.json`, `combo_table.csv`, `winners.json`, `summary.json` and
`chunk_recommendations.json` from the benchmark notebook exports.

- **Cost panel**: real prices for all LLMs, Whisper models and embeddings
- **Benchmark tab**: sorted combo table and sweet-spot cards
- **Illustration tab**: key insights summary


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Benchmark Data Loader
# ═══════════════════════════════════════════════════════════════════

import json as _json
from pathlib import Path as _Path
import pandas as _pd

def _find_benchmark_dir():
    # Search both new (benchmark_imports_for_QandA_notebook) and legacy
    # (benchmark_results) folder names, in both Colab and local locations.
    folder_names = ["benchmark_imports_for_QandA_notebook", "benchmark_results"]
    locations = [
        BASE_DIR,
        _Path("/content"),
        _Path.cwd(),
        _Path("/content/drive/MyDrive"),
    ]
    seen = set()
    for loc in locations:
        for name in folder_names:
            c = (loc / name)
            try:
                c = c.resolve() if c.exists() else c
            except Exception:
                pass
            if c in seen:
                continue
            seen.add(c)
            if c.exists() and (c / "prices.json").exists():
                return c
    return None

# Fallback prices (used when benchmark exports are not available)
_FALLBACK_LLM_PRICES = {
    "gpt-5":          {"input": 1.25, "output": 10.00, "type": "openai"},
    "gpt-5-mini":     {"input": 0.25, "output":  2.00, "type": "openai"},
    "gpt-4o":         {"input": 2.50, "output": 10.00, "type": "openai"},
    "gpt-4o-mini":    {"input": 0.15, "output":  0.60, "type": "openai"},
    "hf:llama-3.3":   {"input": 0.0, "output": 0.0, "type": "huggingface"},
    "hf:qwen2.5-72b": {"input": 0.0, "output": 0.0, "type": "huggingface"},
}

_FALLBACK_WHISPER_PRICES = {
    "api (whisper-1)": {"cost_per_minute_usd": 0.006, "type": "api"},
    "local: tiny":     {"cost_per_minute_usd": 0.0, "type": "local"},
    "local: base":     {"cost_per_minute_usd": 0.0, "type": "local"},
    "local: small":    {"cost_per_minute_usd": 0.0, "type": "local"},
    "local: medium":   {"cost_per_minute_usd": 0.0, "type": "local"},
    "local: large-v3": {"cost_per_minute_usd": 0.0, "type": "local"},
}

_FALLBACK_EMBEDDING_PRICES = {
    "text-embedding-3-small": {"cost_per_1m_tokens_usd": 0.02, "type": "openai"},
    "text-embedding-3-large": {"cost_per_1m_tokens_usd": 0.13, "type": "openai"},
}

BENCHMARK_DATA = {
    "source": "fallback",
    "benchmark_dir": None,
    "llm_prices": dict(_FALLBACK_LLM_PRICES),
    "whisper_prices": dict(_FALLBACK_WHISPER_PRICES),
    "embedding_prices": dict(_FALLBACK_EMBEDDING_PRICES),
    "combo_table": None,
    "winners": None,
    "summary": None,
}

_bdir = _find_benchmark_dir()
if _bdir is not None:
    print(f"📊 Benchmark data found: {_bdir}")
    BENCHMARK_DATA["benchmark_dir"] = str(_bdir)
    BENCHMARK_DATA["source"] = "benchmark_notebook"
    try:
        with open(_bdir / "prices.json") as f:
            prices = _json.load(f)
        if prices.get("llm_models"):
            BENCHMARK_DATA["llm_prices"] = prices["llm_models"]
        if prices.get("whisper_models"):
            BENCHMARK_DATA["whisper_prices"] = prices["whisper_models"]
        if prices.get("embedding_models"):
            BENCHMARK_DATA["embedding_prices"] = prices["embedding_models"]
    except Exception as e:
        print(f"   ✕  prices.json: {e}")
    try:
        BENCHMARK_DATA["combo_table"] = _pd.read_csv(_bdir / "combo_table.csv")
    except Exception as e:
        print(f"   ✕  combo_table.csv: {e}")
    try:
        with open(_bdir / "winners.json") as f:
            BENCHMARK_DATA["winners"] = _json.load(f)
    except Exception as e:
        print(f"   ✕  winners.json: {e}")
    try:
        with open(_bdir / "summary.json") as f:
            BENCHMARK_DATA["summary"] = _json.load(f)
    except Exception as e:
        print(f"   ✕  summary.json: {e}")
    try:
        with open(_bdir / "chunk_recommendations.json") as f:
            BENCHMARK_DATA["chunk_recommendations"] = _json.load(f)
        print(f"   ✅ chunk_recommendations.json loaded")
    except Exception as e:
        print(f"   ✕  chunk_recommendations.json: {e}")
else:
    print("ℹ️  No benchmark outputs found — using fallback defaults")
    BENCHMARK_DATA["chunk_recommendations"] = None

def get_llm_price(model_name: str) -> dict:
    return BENCHMARK_DATA["llm_prices"].get(model_name, {"input": 0.0, "output": 0.0, "type": "openai"})

def get_whisper_price_per_minute(whisper_option: str) -> float:
    info = BENCHMARK_DATA["whisper_prices"].get(whisper_option, {})
    return float(info.get("cost_per_minute_usd", 0.0))

def get_embedding_price_per_1m(model: str = None) -> float:
    if model is None: model = CONFIG.get("embedding_model", "text-embedding-3-small")
    info = BENCHMARK_DATA["embedding_prices"].get(model, {})
    return float(info.get("cost_per_1m_tokens_usd", 0.0))

print(f"💡 Source: {BENCHMARK_DATA['source']}")

# ── Adaptive chunk configuration from benchmark data ──────────────
_CHUNK_RECOMMENDATIONS = BENCHMARK_DATA.get("chunk_recommendations") or {}

def get_chunk_config(duration_sec: int, whisper_option: str = None) -> dict:
    """
    Return adaptive chunk_size and chunk_overlap from chunk_recommendations.json.

    Structure (benchmark_final.ipynb Cell 66):
      outer key = whisper model (api, tiny, base, small, medium, large-v3)
      inner key = video bucket  (short_video, medium_video, long_video)
      values    = chunk_size, overlap (NOT chunk_overlap), mrr
      _buckets  = duration ranges per bucket
    """
    if not _CHUNK_RECOMMENDATIONS:
        raise RuntimeError(
            "chunk_recommendations.json not loaded. "
            f"Place it in {BASE_DIR / 'benchmark_imports_for_QandA_notebook'}/ and re-run."
        )

    buckets = _CHUNK_RECOMMENDATIONS["_buckets"]

    video_bucket = None
    for bucket_name, b_range in buckets.items():
        if b_range["min_sec"] <= duration_sec <= b_range["max_sec"]:
            video_bucket = bucket_name
            break

    if video_bucket is None:
        raise RuntimeError(
            f"duration_sec={duration_sec} does not match any bucket. "
            f"Buckets: {list(buckets.keys())}"
        )

    if whisper_option is None:
        raise RuntimeError("whisper_option must be provided to get_chunk_config.")
    if whisper_option.startswith("local:"):
        whisper_key = whisper_option.replace("local:", "").strip()
    elif "api" in whisper_option:
        whisper_key = "api"
    else:
        raise RuntimeError(
            f"Cannot map whisper_option '{whisper_option}' to a benchmark key."
        )

    if whisper_key not in _CHUNK_RECOMMENDATIONS:
        raise RuntimeError(
            f"Whisper model '{whisper_key}' not found in chunk_recommendations.json. "
            f"Available: {[k for k in _CHUNK_RECOMMENDATIONS if k != '_buckets']}"
        )

    rec = _CHUNK_RECOMMENDATIONS[whisper_key].get(video_bucket)
    if not rec:
        raise RuntimeError(
            f"No recommendation for whisper='{whisper_key}', bucket='{video_bucket}'."
        )

    return {
        "chunk_size":    int(rec["chunk_size"]),
        "chunk_overlap": int(rec["overlap"]),
        "mrr":           float(rec.get("mrr", 0)),
        "source":        "benchmark_data",
        "whisper_key":   whisper_key,
        "bucket":        video_bucket,
    }

print("✅ get_chunk_config() ready (adaptive chunking from benchmark data)")


## 6. Audio Download (yt-dlp)

In [9]:
import re, time
from dataclasses import dataclass, field
from typing import Optional
import yt_dlp

@dataclass
class VideoMetadata:
    video_id: str
    title: str
    channel: str
    duration_sec: int
    url: str

@dataclass
class TranscriptResult:
    video_id: str
    text: str
    backend: str
    whisper_model: str
    language: Optional[str] = None
    whisper_cost_usd: float = 0.0
    transcribe_time_sec: float = 0.0
    # Segments with timestamps
    segments: list = field(default_factory=list)


def extract_video_id(url_or_id: str) -> str:
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", url_or_id):
        return url_or_id
    patterns = [r"(?:v=|/v/|youtu\.be/|/embed/|/shorts/)([A-Za-z0-9_-]{11})"]
    for pat in patterns:
        m = re.search(pat, url_or_id)
        if m:
            return m.group(1)
    raise ValueError(f"could not extract '{url_or_id}'")


def get_video_metadata(url: str) -> VideoMetadata:
    ydl_opts = {"quiet": True, "no_warnings": True, "skip_download": True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
    return VideoMetadata(
        video_id=info["id"],
        title=info.get("title", "Unknown"),
        channel=info.get("uploader", "Unknown"),
        duration_sec=info.get("duration", 0),
        url=f"https://www.youtube.com/watch?v={info['id']}",
    )


def download_audio(video_id: str) -> str:
    url = f"https://www.youtube.com/watch?v={video_id}"
    output_template = os.path.join(CONFIG["audio_dir"], f"{video_id}.%(ext)s")
    ydl_opts = {
        "format": f"bestaudio[ext={CONFIG['audio_format']}]/bestaudio",
        "outtmpl": output_template,
        "quiet": True,
        "no_warnings": True,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": CONFIG["audio_format"],
            "preferredquality": CONFIG["audio_bitrate_kbps"],
        }],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    audio_path = os.path.join(CONFIG["audio_dir"], f"{video_id}.{CONFIG['audio_format']}")
    if not os.path.exists(audio_path):
        raise RuntimeError(f"Audio download failed for {video_id}")
    size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    print(f"   📁 Audio: {size_mb:.2f} MB")
    return audio_path

print("✅ Audio-download ready")


✅ Audio-download ready


## 7. Whisper Transcription (API + Local + User Audio)

In [10]:
from openai import OpenAI

openai_client = OpenAI()
WHISPER_API_COST_PER_MINUTE = 0.006

_local_whisper_cache = {}


def _get_local_whisper_model(model_size: str):
    if model_size not in _local_whisper_cache:
        import whisper as openai_whisper_oss
        import torch
        print(f"   ⬇️  Loading Whisper '{model_size}'...")
        _local_whisper_cache[model_size] = openai_whisper_oss.load_model(model_size)
        print(f"   ✅ Model loaded (GPU: {torch.cuda.is_available()})")
    return _local_whisper_cache[model_size]


def transcribe_with_api(audio_path: str, video_id: str) -> TranscriptResult:
    """OpenAI Whisper API. Extracts segments with timestamps."""
    file_size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    if file_size_mb > CONFIG["whisper_max_mb"]:
        raise RuntimeError(
            f"Audio is {file_size_mb:.1f} MB – exceeding the API-limit."
        )
    start = time.time()
    with open(audio_path, "rb") as f:
        response = openai_client.audio.transcriptions.create(
            model="whisper-1", file=f, response_format="verbose_json",
        )
    elapsed = time.time() - start
    duration_min = (response.duration or 0) / 60

    # Extract segments — API returns a list of objects
    segments = []
    if hasattr(response, "segments") and response.segments:
        for seg in response.segments:
            # seg can be a dict or a Pydantic object
            if hasattr(seg, "text"):
                segments.append({
                    "text": seg.text.strip(),
                    "start": float(seg.start),
                    "end": float(seg.end),
                })
            elif isinstance(seg, dict):
                segments.append({
                    "text": seg["text"].strip(),
                    "start": float(seg["start"]),
                    "end": float(seg["end"]),
                })

    return TranscriptResult(
        video_id=video_id,
        text=response.text,
        backend="api",
        whisper_model="whisper-1",
        language=response.language,
        whisper_cost_usd=duration_min * WHISPER_API_COST_PER_MINUTE,
        transcribe_time_sec=elapsed,
        segments=segments,
    )


def transcribe_with_local(audio_path: str, video_id: str) -> TranscriptResult:
    """Local open-source Whisper. Extracts segments with timestamps."""
    import torch
    model_size = CONFIG["whisper_local_size"]
    model = _get_local_whisper_model(model_size)
    start = time.time()
    result = model.transcribe(audio_path, fp16=torch.cuda.is_available())
    elapsed = time.time() - start
    cost_estimate = (elapsed / 3600) * 0.10

    # Open-Source Whisper delivers segments as a list of dicts with start/end/text
    segments = []
    for seg in result.get("segments", []):
        segments.append({
            "text": seg["text"].strip(),
            "start": float(seg["start"]),
            "end": float(seg["end"]),
        })

    return TranscriptResult(
        video_id=video_id,
        text=result["text"].strip(),
        backend="local",
        whisper_model=model_size,
        language=result.get("language"),
        whisper_cost_usd=cost_estimate,
        transcribe_time_sec=elapsed,
        segments=segments,
    )


def transcribe_audio(audio_path: str, video_id: str) -> TranscriptResult:
    """Dispatch transcription to API or local Whisper based on CONFIG."""
    backend = CONFIG["whisper_backend"]
    if backend == "api":
        print(f"   🎙️  Whisper API...")
        return transcribe_with_api(audio_path, video_id)
    elif backend == "local":
        print(f"   🎙️  Open-Source Whisper ({CONFIG['whisper_local_size']})...")
        return transcribe_with_local(audio_path, video_id)
    else:
        raise ValueError(f"Unknown whisper_backend: '{backend}'")


def transcribe_user_audio(audio_path: str) -> str:
    """Transcribes user microphone audio via the OpenAI Whisper API."""
    file_size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    if file_size_mb > CONFIG["whisper_max_mb"]:
        raise RuntimeError(f"Audio too large: {file_size_mb:.1f} MB (limit: {CONFIG['whisper_max_mb']} MB)")
    with open(audio_path, "rb") as f:
        response = openai_client.audio.transcriptions.create(
            model="whisper-1", file=f,
        )
    return response.text.strip()


print("✅ Whisper pipeline ready (API + local + user audio, segment timestamps)")


✅ Whisper pipeline ready (API + local + user audio, segment timestamps)


## 8. Chunking with tiktoken

In [11]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

_encoding = tiktoken.get_encoding("cl100k_base")

def token_length(text: str) -> int:
    return len(_encoding.encode(text))

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    length_function=token_length,
    separators=["\n\n", "\n", ". ", "? ", "! ", "; ", ", ", " ", ""],
)

def _format_timestamp(seconds: float) -> str:
    """Format seconds as MM:SS or HH:MM:SS."""
    total = int(seconds)
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"

def chunk_segments_with_timestamps(segments: list[dict], chunk_size: int,
                                    chunk_overlap: int) -> list[dict]:
    """
    Segment-aware Chunking: combines Whisper segments into chunks until the
    target token size is reached. Each chunk retains its start and end timestamp.

    Chunk start reflects the beginning of new content, not the overlap section,
    so timestamp deep links (?t=Xs) correctly target the semantic start of each chunk.
    """
    if not segments:
        return []

    chunks = []
    current_segs = []
    current_tokens = 0
    overlap_count = 0

    for seg in segments:
        seg_tokens = token_length(seg["text"])

        if current_tokens + seg_tokens > chunk_size and current_segs:

            new_start_idx = overlap_count if overlap_count < len(current_segs) else 0
            text = " ".join(s["text"] for s in current_segs).strip()
            chunks.append({
                "text": text,
                "start": current_segs[new_start_idx]["start"],
                "end": current_segs[-1]["end"],
                "start_formatted": _format_timestamp(current_segs[new_start_idx]["start"]),
                "end_formatted": _format_timestamp(current_segs[-1]["end"]),
                "num_segments": len(current_segs),
            })

            # Overlap/carry over: Includes the final N segments from the previous chunk.
            overlap_segs = []
            overlap_tokens = 0
            for s in reversed(current_segs):
                s_tokens = token_length(s["text"])
                if overlap_tokens + s_tokens > chunk_overlap:
                    break
                overlap_segs.insert(0, s)
                overlap_tokens += s_tokens
            current_segs = overlap_segs
            current_tokens = overlap_tokens
            overlap_count = len(overlap_segs)

        current_segs.append(seg)
        current_tokens += seg_tokens

    if current_segs and len(current_segs) > overlap_count:
        new_start_idx = overlap_count if overlap_count < len(current_segs) else 0
        text = " ".join(s["text"] for s in current_segs).strip()
        chunks.append({
            "text": text,
            "start": current_segs[new_start_idx]["start"],
            "end": current_segs[-1]["end"],
            "start_formatted": _format_timestamp(current_segs[new_start_idx]["start"]),
            "end_formatted": _format_timestamp(current_segs[-1]["end"]),
            "num_segments": len(current_segs),
        })

    return chunks

def chunk_transcript(transcript: TranscriptResult, metadata: VideoMetadata) -> list[Document]:
    """
    Chunk the transcript. Uses segment-aware chunking if timestamps are available,
    otherwise falls back to plain text splitting.
    """
    documents = []

    if transcript.segments:

        # Timestamp-aware chunking
        chunks = chunk_segments_with_timestamps(
            transcript.segments,
            CONFIG["chunk_size"],
            CONFIG["chunk_overlap"],
        )
        for i, chunk in enumerate(chunks):
            doc = Document(
                page_content=chunk["text"],
                metadata={
                    "video_id": metadata.video_id,
                    "title": metadata.title,
                    "channel": metadata.channel,
                    "url": metadata.url,
                    "duration_sec": metadata.duration_sec,
                    "language": transcript.language or "unknown",
                    "whisper_backend": transcript.backend,
                    "whisper_model": transcript.whisper_model,
                    "chunk_index": i,
                    "total_chunks": len(chunks),

                    # Timestamp metadata
                    "start_sec": chunk["start"],
                    "end_sec": chunk["end"],
                    "start_formatted": chunk["start_formatted"],
                    "end_formatted": chunk["end_formatted"],
                    "num_segments": chunk["num_segments"],
                    "has_timestamps": True,
                }
            )
            documents.append(doc)
    else:
        # Fallback: Text-Splitting without timestamps
        chunks = text_splitter.split_text(transcript.text)
        for i, chunk in enumerate(chunks):
            doc = Document(
                page_content=chunk,
                metadata={
                    "video_id": metadata.video_id,
                    "title": metadata.title,
                    "channel": metadata.channel,
                    "url": metadata.url,
                    "duration_sec": metadata.duration_sec,
                    "language": transcript.language or "unknown",
                    "whisper_backend": transcript.backend,
                    "whisper_model": transcript.whisper_model,
                    "chunk_index": i,
                    "total_chunks": len(chunks),
                    "has_timestamps": False,
                }
            )
            documents.append(doc)

    return documents

print("✅ Chunking ready — adaptive mode (chunk size is set per video at ingest time)")


✅ Chunking ready — adaptive mode (chunk size is set per video at ingest time)


## 9. ChromaDB + Ingestion Pipeline

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=CONFIG["embedding_model"])

# Tracks token usage for ingest and query embedding calls.
# Cost is calculated from prices in BENCHMARK_DATA.

SESSION_EMBEDDING_TOKENS = {"ingest": 0, "query": 0}
SESSION_EMBEDDING_COST_USD = 0.0


# Accurate token counts via tiktoken
import tiktoken as _tk_emb
try:
    _embed_encoder = _tk_emb.encoding_for_model(CONFIG.get("embedding_model", "text-embedding-3-small"))
except KeyError:
    # Fallback for models not in tiktoken's registry
    _embed_encoder = _tk_emb.get_encoding("cl100k_base")


def _count_tokens(text: str) -> int:
    """Return token count for text using tiktoken."""
    if not text:
        return 0
    return len(_embed_encoder.encode(text))


# Alias for backwards compatibility
_estimate_tokens = _count_tokens


def _track_embedding_cost(tokens: int, kind: str = "ingest"):
    """Add embedding cost for the given token count to the session tracker."""
    global SESSION_EMBEDDING_COST_USD
    SESSION_EMBEDDING_TOKENS[kind] = SESSION_EMBEDDING_TOKENS.get(kind, 0) + tokens
    price_per_1m = get_embedding_price_per_1m()
    cost = tokens * price_per_1m / 1_000_000
    SESSION_EMBEDDING_COST_USD += cost
    return cost


# Ein gemeinsamer ChromaDB-Client für ALLE Collections.
# Vorher hatten LangChain-Chroma und chromadb je einen eigenen Client auf
# demselben persist_directory — das kann zu Inkonsistenzen führen, wenn beide
# parallel schreiben. Jetzt wird der Client einmal angelegt und an LangChain
# weitergereicht; die Cache-Collections nutzen denselben Client.
import chromadb as _chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings as _Embeddings


class _NoOpEmbedding(EmbeddingFunction):
    """Dummy embedding function — stores documents as plain text, no vectors."""
    def __init__(self): pass
    def __call__(self, input: Documents) -> _Embeddings:
        return [[0.0] for _ in input]


_chroma_client = _chromadb.PersistentClient(path=CONFIG["chroma_dir"])

# Vector-Collection (mit echten Embeddings) — wird über LangChain genutzt.
vectorstore = Chroma(
    client=_chroma_client,
    collection_name=CONFIG["collection_name"],
    embedding_function=embeddings,
)

# Cache-Collections für rohe Texte (YouTube-Captions, voller Whisper-Transkript).
# Nutzen denselben Client → keine Race Conditions, konsistentes Lock-Verhalten.
_captions_collection = _chroma_client.get_or_create_collection(
    "yt_captions_cache", embedding_function=_NoOpEmbedding()
)
_whisper_text_collection = _chroma_client.get_or_create_collection(
    "whisper_full_text", embedding_function=_NoOpEmbedding()
)


def store_yt_captions(video_id: str, captions_text: str) -> None:
    """Persist raw YouTube captions so we never need to re-fetch them."""
    try:
        _captions_collection.upsert(
            ids=[video_id],
            documents=[captions_text],
            metadatas=[{"video_id": video_id}],
        )
    except Exception as e:
        print(f"   ⚠️  Caption cache write failed: {e}")


def get_cached_yt_captions(video_id: str) -> str | None:
    """Return cached YouTube captions, or None if not yet stored."""
    try:
        result = _captions_collection.get(ids=[video_id])
        if result.get("ids"):
            return result["documents"][0] or None
    except Exception:
        pass
    return None


def store_whisper_full_text(video_id: str, text: str) -> None:
    """Persist the complete, un-chunked Whisper transcript for WER comparison."""
    try:
        _whisper_text_collection.upsert(
            ids=[video_id],
            documents=[text],
            metadatas=[{"video_id": video_id}],
        )
    except Exception as e:
        print(f"   ⚠️  Whisper text cache write failed: {e}")


def get_cached_whisper_text(video_id: str) -> str | None:
    """Return the full Whisper transcript text, or None if not cached."""
    try:
        result = _whisper_text_collection.get(ids=[video_id])
        if result.get("ids"):
            return result["documents"][0] or None
    except Exception:
        pass
    return None

def video_already_indexed(video_id: str) -> bool:
    try:
        existing = vectorstore.get(where={"video_id": video_id}, limit=1)
        return len(existing.get("ids", [])) > 0
    except Exception:
        return False


def cleanup_audio_file(audio_path: str):
    if CONFIG["keep_audio_files"]:
        return
    try:
        if os.path.exists(audio_path):
            os.remove(audio_path)
    except Exception as e:
        print(f"   ⚠️  Audio cleanup failed: {e}")


def ingest_video(url_or_id: str, force: bool = False) -> dict:
    start_total = time.time()
    video_id = extract_video_id(url_or_id)

    if not force and video_already_indexed(video_id):
        return {"status": "already_indexed", "video_id": video_id,
                "message": f"Video {video_id} ist bereits in der Datenbank."}

    print(f"🎬 Ingesting {video_id}")
    metadata = get_video_metadata(f"https://www.youtube.com/watch?v={video_id}")
    print(f"   📺 {metadata.title} ({metadata.duration_sec}s)")

    # Fetch & cache YouTube captions at ingest time (best-effort, does not block ingest)
    _yt_captions = get_youtube_reference_transcript(video_id)
    if _yt_captions:
        store_yt_captions(video_id, _yt_captions)
        print(f"   📝 YouTube captions cached ({len(_yt_captions)} chars)")
    else:
        print(f"   ℹ️  No YouTube captions available for {video_id}")

    audio_path = download_audio(video_id)
    try:
        transcript = transcribe_audio(audio_path, video_id)
        print(f"   ✅ Transkript: {len(transcript.text)} chars, {transcript.language}, {transcript.transcribe_time_sec:.1f}s, ${transcript.whisper_cost_usd:.4f}")
        store_whisper_full_text(video_id, transcript.text)
    finally:
        cleanup_audio_file(audio_path)

    documents = chunk_transcript(transcript, metadata)
    print(f"   📦 {len(documents)} Chunks")

    # Estimate embedding cost before add_documents (ChromaDB embeds implicitly on insert).
    ingest_tokens = sum(_estimate_tokens(d.page_content) for d in documents)
    ingest_embed_cost = _track_embedding_cost(ingest_tokens, kind="ingest")
    print(f"   🔢 Embedding: ~{ingest_tokens} tokens, ${ingest_embed_cost:.5f}")

    vectorstore.add_documents(documents)

    elapsed_total = time.time() - start_total
    return {
        "status": "success",
        "video_id": video_id,
        "title": metadata.title,
        "channel": metadata.channel,
        "duration_sec": metadata.duration_sec,
        "language": transcript.language,
        "whisper_backend": transcript.backend,
        "whisper_model": transcript.whisper_model,
        "num_chunks": len(documents),
        "whisper_cost_usd": transcript.whisper_cost_usd,
        "whisper_time_sec": transcript.transcribe_time_sec,
        "embedding_cost_usd": ingest_embed_cost,
        "embedding_tokens": ingest_tokens,
        "total_ingestion_sec": elapsed_total,
    }

print("✅ Ingestion pipeline ready")


## 10. Ground-Truth Evaluation

Two eval functions used in the UI:

1. **`compare_whisper_vs_captions()`** — WER (Word Error Rate) between Whisper output and YouTube captions
2. **`judge_agent_answer()`** — LLM-as-Judge faithfulness scoring for agent answers


In [13]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "youtube-transcript-api>=0.6.0", "--upgrade", "-q"], check=True)
print("✅ youtube-transcript-api upgraded")

✅ youtube-transcript-api upgraded


In [14]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
import json as _json
import re


def get_youtube_reference_transcript(video_id: str, use_cache: bool = True):
    """Gets official YouTube-Captions. Checks local cache first, falls back to live API."""
    # 1. Try cache
    if use_cache:
        cached = get_cached_yt_captions(video_id)
        if cached:
            print(f"[Captions] ✅ Loaded from cache ({len(cached)} chars)")
            return cached

    # 2. Live fetch from YouTube
    try:
        transcript_list = YouTubeTranscriptApi().list(video_id)
        transcript = None
        for t in transcript_list:
            transcript = t
            break
        if transcript is None:
            return None
        snippets = transcript.fetch()
        text = " ".join(s.text.replace("\n", " ") for s in snippets)
        result = re.sub(r"\s+", " ", text).strip()
        # Persist for next time
        if result:
            store_yt_captions(video_id, result)
            print(f"[Captions] 📥 Fetched live + cached ({len(result)} chars)")
        return result
    except Exception as e:
        print(f"[Captions] Error: {type(e).__name__}: {e}")
        return None


def normalize_for_wer(text: str) -> str:
    """
    Normalise a transcript for fairer WER comparison.

    Removes superficial divergence between Whisper output and YouTube
    auto-captions that inflates WER without reflecting transcription quality:

      1. Sound / music tags   [Music], [Applause], [Laughter]
      2. Speaker labels       >> SPEAKER_01:  or  (Speaker):
      3. Stutters / repairs   word-  (trailing dash before space)
      4. Disfluencies         um, uh, hmm, you know what i mean, you know
      5. Contractions         it's -> it is, don't -> do not, etc.
      6. Hyphens in compounds real-time -> real time
      7. Punctuation          removed entirely
      8. Lowercase + collapse whitespace
    """
    # 1. Sound / annotation tags
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    # 2. Speaker labels  >> NAME:  or  NAME:
    text = re.sub(r'>>\s*\S+\s*:', '', text)
    # 3. Stutters: word- <space>
    text = re.sub(r'\b\w+-(?=\s)', '', text)
    # 4. Disfluencies (conservative - leave "like" untouched)
    text = re.sub(
        r'\b(um+|uh+|uh-huh|hmm+|mhm|you know what i mean|you know)\b',
        '', text, flags=re.IGNORECASE,
    )
    # 5. Contractions
    _contractions = {
        "it's": "it is", "that's": "that is", "i'm": "i am",
        "you're": "you are", "they're": "they are", "we're": "we are",
        "he's": "he is", "she's": "she is", "don't": "do not",
        "doesn't": "does not", "didn't": "did not", "can't": "cannot",
        "won't": "will not", "isn't": "is not", "aren't": "are not",
        "wasn't": "was not", "weren't": "were not", "i've": "i have",
        "you've": "you have", "we've": "we have", "they've": "they have",
        "i'll": "i will", "you'll": "you will", "we'll": "we will",
        "they'll": "they will", "i'd": "i would", "you'd": "you would",
        "we'd": "we would", "they'd": "they would", "it'll": "it will",
        "let's": "let us", "that'll": "that will", "there's": "there is",
        "here's": "here is", "what's": "what is", "who's": "who is",
        "how's": "how is", "where's": "where is", "when's": "when is",
        "that'd": "that would", "he'd": "he would", "she'd": "she would",
        "could've": "could have", "would've": "would have",
        "should've": "should have",
    }
    for contraction, expanded in _contractions.items():
        text = re.sub(
            r'\b' + re.escape(contraction) + r'\b',
            expanded, text, flags=re.IGNORECASE,
        )
    # 6. Hyphens in compound words -> space
    text = text.replace('-', ' ')
    # 7. Remove all remaining punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # 8. Lowercase and collapse whitespace
    return re.sub(r'\s+', ' ', text.lower()).strip()


def compare_whisper_vs_captions(video_id: str) -> dict:
    """Compares Whisper-Transkript with YouTube-Captions via WER.
    Uses the original full Whisper text (stored at ingest time) — not chunk
    reconstruction — to avoid double-counting overlap regions that inflate WER.
    Both sides are normalised with normalize_for_wer() before scoring.
    """
    # Use the full, un-chunked Whisper transcript stored at ingest time
    whisper_text = get_cached_whisper_text(video_id)
    if not whisper_text:
        # Fallback: reconstruct from chunks (deduplicated by chunk_index)
        docs = vectorstore.get(where={"video_id": video_id})
        if not docs or not docs.get("documents"):
            return {"status": "error", "message": f"Video {video_id} not found in the database."}
        metas = docs["metadatas"]
        texts = docs["documents"]
        # Deduplicate: keep only first occurrence of each chunk_index
        seen_idx = set()
        ordered = []
        for meta, text in sorted(zip(metas, texts), key=lambda x: x[0].get("chunk_index", 0)):
            ci = meta.get("chunk_index", 0)
            if ci not in seen_idx:
                seen_idx.add(ci)
                ordered.append(text)
        whisper_text = " ".join(ordered)

    reference = get_youtube_reference_transcript(video_id)
    if reference is None:
        return {"status": "no_captions",
                "message": "No YouTube-Captions available for this video."}

    whisper_norm = normalize_for_wer(whisper_text)
    reference_norm = normalize_for_wer(reference)

    try:
        from jiwer import wer
        error_rate = wer(reference_norm, whisper_norm)
    except Exception as e:
        return {"status": "error", "message": f"WER-Calculation failed: {e}"}

    return {
        "status": "success",
        "video_id": video_id,
        "wer_percent": round(error_rate * 100, 2),
        "whisper_length": len(whisper_norm.split()),
        "reference_length": len(reference_norm.split()),
        "whisper_preview": whisper_norm[:400] + "...",
        "reference_preview": reference_norm[:400] + "...",
    }


def judge_agent_answer(question: str, agent_answer: str, video_id: str) -> dict:
    """LLM-as-Judge: checks Faithfulness of the Agents answer with YouTube-Captions."""
    reference = get_youtube_reference_transcript(video_id)
    if reference is None:
        return {"status": "no_captions",
                "message": "Keine Referenz-Captions für Faithfulness-Eval verfügbar."}

    max_chars = 12000
    if len(reference) > max_chars:
        reference = reference[:max_chars] + "..."

    judge_prompt = f'''You are a careful fact-checker. Assess whether an AI assistant's answer is supported by the provided reference transcript.

REFERENCE TRANSCRIPT (ground truth):
"""{reference}"""

USER QUESTION:
{question}

AI ASSISTANT'S ANSWER:
{agent_answer}

Respond with ONLY valid JSON (no markdown, no extra text):
{{
  "faithfulness_score": <integer 1-5, where 5 = fully supported, 1 = contradicted/hallucinated>,
  "supported_claims": [<strings: claims in the answer that ARE in the reference>],
  "unsupported_claims": [<strings: claims NOT in the reference>],
  "verdict": "<one of: fully_supported, mostly_supported, partially_supported, unsupported, contradicted>",
  "explanation": "<one sentence summary>"
}}'''

    from langchain_openai import ChatOpenAI
    judge_llm = ChatOpenAI(model=CONFIG["helper_model"], temperature=0)
    try:
        response = judge_llm.invoke(judge_prompt)
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```(?:json)?\n?", "", raw)
            raw = re.sub(r"\n?```$", "", raw)
        verdict = _json.loads(raw)
        return {"status": "success", "video_id": video_id,
                "judge_model": CONFIG["helper_model"], **verdict}
    except Exception as e:
        return {"status": "error", "message": f"Judge-Call fehlgeschlagen: {e}"}


print("✅ Ground-Truth Eval-Tools bereit:")
print("   • compare_whisper_vs_captions(video_id)")
print("   • judge_agent_answer(question, answer, video_id)")

✅ Ground-Truth Eval-Tools bereit:
   • compare_whisper_vs_captions(video_id)
   • judge_agent_answer(question, answer, video_id)


In [15]:
# DEBUG: Test caption fetch directly
test_vid = "l-rBd0F6d9E"
result = get_youtube_reference_transcript(test_vid)
print("Result:", result[:200] if result else "None — check output above for [Captions] error")

[Captions] 📥 Fetched live + cached (3515 chars)
Result: Most people think The Art of War is about fighting, But it is not. The greatest teaching from Sun Tzu is how to win without fighting at all. And once you understand this, you begin to see conflict ver


In [16]:
from youtube_transcript_api import YouTubeTranscriptApi
print([m for m in dir(YouTubeTranscriptApi) if not m.startswith('_')])

['fetch', 'list']


## 11. LangChain Agent Tools (7)

| Tool | Purpose |
|------|---------|
| `search_video_content` | Semantic search over transcript chunks |
| `list_indexed_videos` | List all videos in the vector store |
| `add_new_video` | Download, transcribe and index a new video |
| `get_video_metadata_tool` | Retrieve video metadata |
| `summarize_video` | Generate a video summary |
| `compare_videos` | Compare two indexed videos |
| `extract_key_moments` | Extract key moments with timestamps |


In [17]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

helper_llm = ChatOpenAI(model=CONFIG["helper_model"], temperature=0)


@tool
def search_video_content(query: str) -> str:
    """Search the indexed YouTube video transcripts. Use for factual questions.

    Args:
        query: Natural language question or search phrase.
    """
    _track_embedding_cost(_estimate_tokens(query), kind="query")
    # Filter is read directly from active state — agent does not need to pass video_id
    video_id = _current_agent.get("active_video_id")
    if video_id:
        results = vectorstore.similarity_search(
            query,
            k=CONFIG["retrieval_k"],
            filter={"video_id": video_id},
        )
    else:
        results = vectorstore.similarity_search(query, k=CONFIG["retrieval_k"])
    if not results:
        if video_id:
            return f"No relevant content found for video '{video_id}'. The video may not be indexed yet."
        return "No relevant content found. Ask the user to add a video first."
    formatted = []
    for i, doc in enumerate(results, 1):
        meta = doc.metadata
        timestamp_info = ""
        if meta.get("has_timestamps"):
            timestamp_info = f"Timestamp: {meta.get('start_formatted', '?')} - {meta.get('end_formatted', '?')}\n"
        formatted.append(
            f"[Passage {i}]\n"
            f"Video: {meta.get('title', 'Unknown')}\n"
            f"Channel: {meta.get('channel', 'Unknown')}\n"
            f"Language: {meta.get('language', 'unknown')}\n"
            f"{timestamp_info}"
            f"Content: {doc.page_content}\n"
        )
    return "\n---\n".join(formatted)


@tool
def list_indexed_videos() -> str:
    """List all YouTube videos currently indexed."""
    try:
        all_docs = vectorstore.get()
        if not all_docs or not all_docs.get("metadatas"):
            return "No videos indexed yet."
        seen = {}
        for meta in all_docs["metadatas"]:
            vid = meta.get("video_id")
            if vid and vid not in seen:
                seen[vid] = meta
        if not seen:
            return "No videos indexed yet."
        lines = [f"{len(seen)} video(s) indexed:"]
        for i, (vid, meta) in enumerate(seen.items(), 1):
            lines.append(f"{i}. {meta.get('title')} ({meta.get('channel')}, {meta.get('language')})")
        return "\n".join(lines)
    except Exception as e:
        return f"Error: {e}"


@tool
def add_new_video(youtube_url: str) -> str:
    """Add a new YouTube video. Downloads audio, transcribes with Whisper, indexes in DB.

    Args:
        youtube_url: Full YouTube URL or 11-char video ID.
    """
    try:
        result = ingest_video(youtube_url)
        if result["status"] == "already_indexed":
            return result["message"]
        return (
            f"Successfully added:\n"
            f"- Title: {result['title']}\n"
            f"- Channel: {result['channel']}\n"
            f"- Duration: {result['duration_sec']}s\n"
            f"- Language: {result['language']}\n"
            f"- Whisper: {result['whisper_backend']} ({result['whisper_model']})\n"
            f"- Chunks: {result['num_chunks']}\n"
            f"- Cost: ${result['whisper_cost_usd']:.4f}"
        )
    except Exception as e:
        return f"Failed to add video: {e}"


@tool
def get_video_metadata_tool(video_id_or_url: str) -> str:
    """Get metadata for a YouTube video (indexed or not).

    Args:
        video_id_or_url: Video ID or URL.
    """
    try:
        vid = extract_video_id(video_id_or_url)
        existing = vectorstore.get(where={"video_id": vid}, limit=1)
        if existing and existing.get("metadatas"):
            meta = existing["metadatas"][0]
            return (
                f"Title: {meta.get('title')}\n"
                f"Channel: {meta.get('channel')}\n"
                f"Duration: {meta.get('duration_sec')}s\n"
                f"Language: {meta.get('language')}\n"
                f"URL: {meta.get('url')}\n"
                f"Chunks: {meta.get('total_chunks')}"
            )
        m = get_video_metadata(f"https://www.youtube.com/watch?v={vid}")
        return f"[Not indexed] {m.title} | {m.channel} | {m.duration_sec}s | {m.url}"
    except Exception as e:
        return f"Error: {e}"


# --- Extended tools ---

def _get_full_transcript(video_id: str):
    docs = vectorstore.get(where={"video_id": video_id})
    if not docs or not docs.get("documents"):
        return None, None
    metas = docs["metadatas"]
    texts = docs["documents"]
    ordered = sorted(zip(metas, texts), key=lambda x: x[0].get("chunk_index", 0))
    full_text = " ".join(t for _, t in ordered)
    return full_text, (metas[0] if metas else {})


@tool
def summarize_video(video_id_or_url: str = None) -> str:
    """Generate a concise summary of an entire indexed video.
    Use for "summarize", "what is this video about", "give me an overview" questions.

    Args:
        video_id_or_url: Video ID or full YouTube URL of an indexed video.
                         If the active video is known from your instructions, always pass its ID here.
    """
    try:
        if not video_id_or_url or not str(video_id_or_url).strip():
            try:
                all_docs = vectorstore.get()
                ids = list({m.get("video_id") for m in all_docs.get("metadatas", []) if m.get("video_id")})
            except Exception:
                ids = []
            if not ids:
                return "No videos are indexed yet. Ask the user to add a video first."
            if len(ids) == 1:
                video_id_or_url = ids[0]
            else:
                return f"Multiple videos are indexed: {ids}. Please specify which video to summarize."

        vid = extract_video_id(str(video_id_or_url).strip())
        full_text, meta = _get_full_transcript(vid)
        if full_text is None:
            return (
                f"Video '{vid}' is not indexed yet. "
                f"Use the add_new_video tool or ask the user to index it first."
            )
        if len(full_text) > 20000:
            full_text = full_text[:20000] + "..."
        prompt = (
            f"Summarize this YouTube video transcript in 5-7 concise bullet points. "
            f"Focus on main topics, arguments, and notable facts.\n\n"
            f"Video: {meta.get('title', 'Unknown')} ({meta.get('channel', 'Unknown')})\n\n"
            f"Transcript:\n{full_text}\n\n"
            f"Respond with only bullet points, no preamble."
        )
        response = helper_llm.invoke(prompt)
        return f"**Summary of '{meta.get('title')}':**\n\n{response.content}"
    except Exception as e:
        return f"Error summarizing: {e}"


@tool
def compare_videos(video_ids_or_urls: str) -> str:
    """Compare content of multiple indexed videos. Use when the user mentions multiple videos.

    Args:
        video_ids_or_urls: Comma-separated video IDs or URLs (at least 2).
    """
    try:
        ids = [extract_video_id(s.strip()) for s in video_ids_or_urls.split(",") if s.strip()]
        if len(ids) < 2:
            return "Please provide at least 2 video IDs or URLs, comma-separated."
        sections = []
        for vid in ids:
            full_text, meta = _get_full_transcript(vid)
            if full_text is None:
                return f"Video {vid} is not indexed."
            if len(full_text) > 6000:
                full_text = full_text[:6000] + "..."
            sections.append(f"=== {meta.get('title', vid)} ({meta.get('channel', '?')}) ===\n{full_text}")
        all_content = "\n\n".join(sections)
        prompt = (
            f"Compare these {len(ids)} YouTube video transcripts. Identify:\n"
            f"1. Common themes\n"
            f"2. Key differences\n"
            f"3. Unique insights in each\n\n"
            f"{all_content}\n\n"
            f"Provide a structured comparison in 3-5 paragraphs."
        )
        response = helper_llm.invoke(prompt)
        return response.content
    except Exception as e:
        return f"Error comparing: {e}"


@tool
def extract_key_moments(video_id_or_url: str) -> str:
    """Extract the most important moments from an indexed video. Use for "highlights" or "main points" questions.

    Args:
        video_id_or_url: Video ID or URL of an indexed video.
    """
    try:
        vid = extract_video_id(video_id_or_url)
        full_text, meta = _get_full_transcript(vid)
        if full_text is None:
            return f"Video {vid} is not indexed."
        if len(full_text) > 15000:
            full_text = full_text[:15000] + "..."
        prompt = (
            f"From this video transcript, extract the 5-8 most important moments.\n"
            f"For each: a short heading (3-6 words) + 1-2 sentence description.\n\n"
            f"Video: {meta.get('title', 'Unknown')}\n\n"
            f"{full_text}\n\n"
            f"Format as a numbered list."
        )
        response = helper_llm.invoke(prompt)
        return f"Key moments from '{meta.get('title')}':\n\n{response.content}"
    except Exception as e:
        return f"Error: {e}"


AGENT_TOOLS = [
    search_video_content,
    list_indexed_videos,
    add_new_video,
    get_video_metadata_tool,
    summarize_video,
    compare_videos,
    extract_key_moments,
]

print(f"✅ {len(AGENT_TOOLS)} agent tools registered:")
for t in AGENT_TOOLS:
    print(f"   • {t.name}")


✅ 7 agent tools registered:
   • search_video_content
   • list_indexed_videos
   • add_new_video
   • get_video_metadata_tool
   • summarize_video
   • compare_videos
   • extract_key_moments


## 12. LangChain Agent with Memory

Model-agnostic agent built via a factory function — the model can be switched at runtime without restarting the session.


In [18]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
import os

SYSTEM_PROMPT_TEMPLATE = """You are a helpful multilingual assistant specialized in answering questions about YouTube videos.

You have access to tools that let you search indexed video transcripts, retrieve metadata, summarize content, and compare videos.

{active_video_context}

Guidelines:
- Always base your answers on the indexed video content, not general knowledge.
- IMPORTANT: If an active video is specified above, ALWAYS pass its video_id to search_video_content and summarize_video. Never search across all videos when a specific video is active.
- If no video is indexed yet, call list_indexed_videos first to check — do NOT ask the user to provide a URL.
- Respond in the same language the user writes in.
- Be concise and factual. Cite timestamps when available.
- If you cannot find the answer in the indexed content, say so clearly.
- Only show retrieved passages if the user explicitly asks for them.
"""


def build_system_prompt(active_video_id: str = None, active_video_title: str = None) -> str:
    if active_video_id:
        ctx = (
            f"ACTIVE VIDEO: Restrict ALL searches and answers to this video only.\n"
            f"  Title: {active_video_title or active_video_id}\n"
            f"  ID:    {active_video_id}\n"
            f"Always pass video_id='{active_video_id}' when calling search_video_content or summarize_video."
        )
    else:
        ctx = (
            "No specific video is selected. "
            "Call list_indexed_videos first to see what is available, "
            "then answer based on the indexed content. "
            "Do NOT ask the user to provide a video URL."
        )
    return SYSTEM_PROMPT_TEMPLATE.format(active_video_context=ctx)


# Keep SYSTEM_PROMPT as alias for backward compatibility (used by build_agent default)
SYSTEM_PROMPT = build_system_prompt()

memory = MemorySaver()
_current_agent = {"agent": None, "model": None, "provider": None, "active_video_id": None}

def _resolve_model_provider(model_name: str) -> tuple[str, str]:
    """
    Auto-detect provider based on model name prefix.
    - "hf:..." -> HuggingFace Inference API
    - all other -> OpenAI
    """
    if model_name.startswith("hf:"):
        hf_id = CONFIG.get("hf_model_map", {}).get(model_name)
        if hf_id is None:
            raise ValueError(f"Unknown HuggingFace model key: '{model_name}' — add it to CONFIG['hf_model_map']")
        return hf_id, "huggingface"
    else:
        return model_name, "openai"


def build_agent(model_name: str = None, active_video_id: str = None, active_video_title: str = None):
    global memory
    if model_name is None:
        model_name = CONFIG["agent_model"]

    resolved_id, provider = _resolve_model_provider(model_name)

    if provider == "openai":
        llm = ChatOpenAI(model=resolved_id, temperature=0)
    elif provider == "huggingface":
        if not os.environ.get("HF_TOKEN"):
            raise RuntimeError("HF_TOKEN not set — cannot use HuggingFace models.")
        llm = ChatOpenAI(
            model=resolved_id,
            openai_api_key=os.environ["HF_TOKEN"],
            openai_api_base="https://api-inference.huggingface.co/v1",
            temperature=0.01,
            max_tokens=512,
        )

    prompt = build_system_prompt(active_video_id, active_video_title)
    agent = create_react_agent(
        model=llm,
        tools=AGENT_TOOLS,
        prompt=prompt,
        checkpointer=memory,
    )
    _current_agent["agent"] = agent
    _current_agent["model"] = model_name
    _current_agent["provider"] = provider
    _current_agent["active_video_id"] = active_video_id
    return agent

def ask_agent_with_usage(question: str, thread_id: str = "default", model_name: str = None,
                          active_video_id: str = None, active_video_title: str = None) -> dict:
    needs_rebuild = (
        _current_agent["agent"] is None
        or (model_name and model_name != _current_agent["model"])
        or active_video_id != _current_agent.get("active_video_id")
    )
    if needs_rebuild:
        build_agent(model_name or _current_agent.get("model"), active_video_id, active_video_title)

    config = {"configurable": {"thread_id": thread_id}}
    from langchain_community.callbacks import get_openai_callback

    with get_openai_callback() as cb:
        result = _current_agent["agent"].invoke(
            {"messages": [{"role": "user", "content": question}]},
            config=config,
        )

    answer = result["messages"][-1].content
    price_info = get_llm_price(_current_agent["model"])
    cost_usd = (cb.prompt_tokens * price_info.get("input", 0.0) +
                cb.completion_tokens * price_info.get("output", 0.0)) / 1_000_000

    return {
        "answer": answer,
        "input_tokens": cb.prompt_tokens,
        "output_tokens": cb.completion_tokens,
        "cost_usd": cost_usd,
    }

def ask_agent(question: str, thread_id: str = "default", active_video_id: str = None) -> str:
    return ask_agent_with_usage(question, thread_id, active_video_id=active_video_id)["answer"]

try:
    build_agent()
    print(f"✅ Agent ready (model: {_current_agent['model']})")
except NameError:
    print("⚠️  AGENT_TOOLS not yet defined — run the tools cell first.")


✅ Agent ready (model: gpt-4o-mini)


In [19]:
# --- Helper functions for top 5 passages with timestamps used in the UI ---

def get_top_passages_with_metadata(query: str, k: int = 5,
                                     video_id: Optional[str] = None) -> list[dict]:
    """
    Retrieve the top-k most relevant passages for a query,
    structured with timestamps and chunk indices.

    Args:
        query: The search query
        k: Number of results to return
        video_id: Optional — restrict search to this video
    """
    where_filter = {"video_id": video_id} if video_id else None

    # Track query-embedding cost
    _track_embedding_cost(_estimate_tokens(query), kind="query")

    # ChromaDB returns (doc, score) tuples
    results = vectorstore.similarity_search_with_score(
        query, k=k, filter=where_filter
    )

    passages = []
    for doc, score in results:
        meta = doc.metadata
        passages.append({
            "rank": len(passages) + 1,
            "score": float(score),
            "content": doc.page_content,
            "video_id": meta.get("video_id"),
            "title": meta.get("title"),
            "channel": meta.get("channel"),
            "chunk_index": meta.get("chunk_index"),
            "total_chunks": meta.get("total_chunks"),
            "start_sec": meta.get("start_sec"),
            "end_sec": meta.get("end_sec"),
            "start_formatted": meta.get("start_formatted"),
            "end_formatted": meta.get("end_formatted"),
            "has_timestamps": meta.get("has_timestamps", False),
            "url": meta.get("url"),
        })
    return passages


def format_passages_as_markdown(passages: list[dict], show_timestamps: bool = True) -> str:
    """Formats passages as Markdown for the UI."""
    if not passages:
        return "*No relevant passages found.*"

    lines = [f"### 🔍 Top-{len(passages)} relevant passages\n"]
    for p in passages:
        header = f"**{p['rank']}. [{p['title']}]({p['url']})** — *{p['channel']}*"

        if show_timestamps and p["has_timestamps"]:
            # YouTube-URL mit Timestamp: &t=123s
            start_sec = int(p["start_sec"]) if p["start_sec"] else 0
            ts_url = f"{p['url']}&t={start_sec}s"
            header += f"\n⏱️  [{p['start_formatted']} - {p['end_formatted']}]({ts_url})"
        elif show_timestamps and not p["has_timestamps"]:
            header += f"\n⏱️  *(no timestamps available — video indexed before v4)*"
        else:
            header += f"\n🧩 Chunk {p['chunk_index']+1}/{p['total_chunks']}"

        header += f"\n_Score: {p['score']:.3f}_"
        lines.append(header)
        lines.append(f"> {p['content'][:350]}...")
        lines.append("")

    return "\n".join(lines)


print("✅ Passage-Helper functions ready:")
print("   • get_top_passages_with_metadata(query, k=5, video_id=None)")
print("   • format_passages_as_markdown(passages, show_timestamps=True)")


✅ Passage-Helper functions ready:
   • get_top_passages_with_metadata(query, k=5, video_id=None)
   • format_passages_as_markdown(passages, show_timestamps=True)


## 13. Smoke Test

Paste a YouTube URL here to test the full pipeline before launching the UI.


In [20]:
# ═══════════════════════════════════════════════════════════════════
# YouTube-Bot-Block Workaround
# ═══════════════════════════════════════════════════════════════════

import subprocess, sys, importlib

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "--force-reinstall",
     "yt-dlp"],
    check=True,
)


import yt_dlp
importlib.reload(yt_dlp)
print(f"✅ yt-dlp version: {yt_dlp.version.__version__}")


_ROBUST_OPTS = {
    "extractor_args": {
      "youtube": {
            "player_client": ["tv_embedded", "web_safari", "ios", "android"],
            "player_skip": ["webpage", "configs"],
        }
    },
    "http_headers": {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/605.1.15 (KHTML, like Gecko) "
            "Version/17.0 Safari/605.1.15"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    },
    "retries": 5,
    "fragment_retries": 5,
    "extractor_retries": 5,
    "sleep_interval_requests": 1,
}


def get_video_metadata(url: str):
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
        **_ROBUST_OPTS,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
    return VideoMetadata(
        video_id=info["id"],
        title=info.get("title", "Unknown"),
        channel=info.get("uploader", "Unknown"),
        duration_sec=info.get("duration", 0),
        url=f"https://www.youtube.com/watch?v={info['id']}",
    )


def download_audio(video_id: str) -> str:
    url = f"https://www.youtube.com/watch?v={video_id}"
    output_template = os.path.join(CONFIG["audio_dir"], f"{video_id}.%(ext)s")
    ydl_opts = {
        "format": "bestaudio/best",  # ffmpeg converts to m4a via postprocessor
        "outtmpl": output_template,
        "quiet": True,
        "no_warnings": True,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": CONFIG["audio_format"],
            "preferredquality": CONFIG["audio_bitrate_kbps"],
        }],
        **_ROBUST_OPTS,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    audio_path = os.path.join(CONFIG["audio_dir"], f"{video_id}.{CONFIG['audio_format']}")
    if not os.path.exists(audio_path):
        raise RuntimeError(f"Audio download failed for {video_id}")
    size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    print(f"   📁 Audio: {size_mb:.2f} MB")
    return audio_path


print("✅ yt-dlp patched with anti-bot headers and robust extractor options")

✅ yt-dlp version: 2026.03.17
✅ yt-dlp patched with anti-bot headers and robust extractor options


In [21]:
# ═══════════════════════════════════════════════════════════
# 👇 Paste YOUTUBE-LINK here 👇
# ═══════════════════════════════════════════════════════════

TEST_VIDEO_URL = "https://www.youtube.com/watch?v=CYvjC94jDu4"

print("Smoke test — ingesting:", TEST_VIDEO_URL)
print("=" * 60)
result = ingest_video(TEST_VIDEO_URL)
print()
print(result)


Smoke test — ingesting: https://www.youtube.com/watch?v=CYvjC94jDu4
🎬 Ingesting CYvjC94jDu4
   📺 The Only Dating Advice You'll Ever Need! (303s)
[Captions] 📥 Fetched live + cached (5194 chars)
   📝 YouTube captions cached (5194 chars)
   📁 Audio: 4.67 MB
   🎙️  Whisper API...
   ✅ Transkript: 5184 chars, english, 13.3s, $0.0303
   📦 3 Chunks
   🔢 Embedding: ~1305 tokens, $0.00003

{'status': 'success', 'video_id': 'CYvjC94jDu4', 'title': "The Only Dating Advice You'll Ever Need!", 'channel': 'Psych2Go', 'duration_sec': 303, 'language': 'english', 'whisper_backend': 'api', 'whisper_model': 'whisper-1', 'num_chunks': 3, 'whisper_cost_usd': 0.030304000854492186, 'whisper_time_sec': 13.283586502075195, 'embedding_cost_usd': 2.61e-05, 'embedding_tokens': 1305, 'total_ingestion_sec': 33.91563153266907}


In [22]:
# ═══════════════════════════════════════════════════════════
# Auto-Indexing: Benchmark videos ingested at startup
# Add or remove URLs as needed.
# ═══════════════════════════════════════════════════════════

AUTO_INDEX_URLS = [
    "https://www.youtube.com/watch?v=CYvjC94jDu4",   # Smoke-test video
    "https://www.youtube.com/watch?v=l-rBd0F6d9E",   # Benchmark short (~5 min)
    "https://www.youtube.com/watch?v=ubjW4aq_MAg",   # Benchmark medium (~17 min)
]

for _url in AUTO_INDEX_URLS:
    print(f"\n▶️  Auto-indexing: {_url}")
    try:
        _r = ingest_video(_url)
        if _r["status"] == "already_indexed":
            print(f"   ℹ️  Already in database")
        else:
            print(f"   ✅ {_r['title'][:50]} — {_r['num_chunks']} chunks")
    except Exception as _e:
        print(f"   ❌ {_e}")



▶️  Auto-indexing: https://www.youtube.com/watch?v=CYvjC94jDu4
   ℹ️  Already in database

▶️  Auto-indexing: https://www.youtube.com/watch?v=l-rBd0F6d9E
🎬 Ingesting l-rBd0F6d9E
   📺 The Art of War’s Greatest Secret (Win Without Fighting) (328s)
[Captions] ✅ Loaded from cache (3515 chars)
   📝 YouTube captions cached (3515 chars)
   📁 Audio: 3.81 MB
   🎙️  Whisper API...
   ✅ Transkript: 3504 chars, english, 9.3s, $0.0328
   📦 2 Chunks
   🔢 Embedding: ~833 tokens, $0.00002
   ✅ The Art of War’s Greatest Secret (Win Without Figh — 2 chunks

▶️  Auto-indexing: https://www.youtube.com/watch?v=ubjW4aq_MAg
🎬 Ingesting ubjW4aq_MAg
   📺 how to *literally* motivate yourself to do anything (1037s)
[Captions] 📥 Fetched live + cached (16949 chars)
   📝 YouTube captions cached (16949 chars)
   📁 Audio: 12.04 MB
   🎙️  Whisper API...
   ✅ Transkript: 16747 chars, english, 45.6s, $0.1037
   📦 9 Chunks
   🔢 Embedding: ~4265 tokens, $0.00009
   ✅ how to *literally* motivate yourself to do anythin — 9

In [23]:
# DEBUG: Test caption fetch directly
test_vid = "l-rBd0F6d9E"
result = get_youtube_reference_transcript(test_vid)
print("Result:", result[:200] if result else "None — check output above for [Captions] error")

[Captions] ✅ Loaded from cache (3515 chars)
Result: Most people think The Art of War is about fighting, But it is not. The greatest teaching from Sun Tzu is how to win without fighting at all. And once you understand this, you begin to see conflict ver


In [24]:
build_agent(CONFIG["agent_model"])

print("=" * 60)
print("Q&A TEST 1: Content question")
print("=" * 60)
_test_vid = extract_video_id(TEST_VIDEO_URL)
answer = ask_agent("What is the main message of the video?", thread_id="smoke_test", active_video_id=_test_vid)
print(answer)

Q&A TEST 1: Content question


The main message of the video "The Only Dating Advice You'll Ever Need!" emphasizes the importance of respect and communication in building healthy relationships. Key points include:

- Allowing intimacy to develop naturally without overwhelming your date.
- Being open to dating different types of people beyond shared interests.
- Focusing on honesty about your current self rather than past relationships.
- Staying true to yourself and not altering your personality for others.
- Ensuring mutual engagement in conversations.
- Reflecting on your reasons for being in a relationship, prioritizing genuine happiness over external pressures.


In [25]:
from langchain_core.utils.function_calling import convert_to_openai_tool

# Sanity check: validate tool schemas in OpenAI function-calling format.
# Raises an error if any tool definition is broken (e.g. invalid JSON schema).
# The result is not used further — the agent was already built in the cell above.
try:
    _ = [convert_to_openai_tool(t) for t in AGENT_TOOLS]
    print("✅ Tool schemas valid (OpenAI function-calling format).")
except Exception as e:
    print(f"❌ Tool definition error: {e}")


✅ Tool schemas valid (OpenAI function-calling format).


In [26]:
print("=" * 60)
print("Q&A TEST 2: Memory Follow-up")
print("=" * 60)
answer = ask_agent("What specific advice does the video give about communication in relationships?", thread_id="smoke_test", active_video_id=_test_vid)
print(answer)


Q&A TEST 2: Memory Follow-up


The video provides specific advice about communication in relationships, highlighting several key points:

1. **Honesty**: It's crucial to be honest with your date about your feelings. If something makes you uneasy, express it. If you enjoyed your time together, let them know instead of playing it cool (Timestamp: 2:15 - 4:01).

2. **Focus on the Present**: Early in dating, concentrate on who you are now rather than discussing your past. It's important for your date to get to know the current you, rather than hearing about past mistakes or exes (Timestamp: 2:15 - 4:01).

3. **Be Yourself**: Don't alter your personality or appearance to fit someone else's expectations. Authenticity is key to attracting the right person (Timestamp: 2:15 - 4:01).

4. **Mutual Engagement**: Ensure that both parties are engaged in the conversation. Avoid dominating the discussion; instead, ask your date how they feel and allow them to ask questions too. This helps in assessing compatibility (Timestamp: 4:01

In [27]:
print("=" * 60)
print("Q&A TEST 3: Multilingual (German)")
print("=" * 60)
answer = ask_agent("Fasse die Kernbotschaft auf Deutsch zusammen.", thread_id="smoke_test", active_video_id=_test_vid)
print(answer)


Q&A TEST 3: Multilingual (German)


Die Kernbotschaft des Videos "The Only Dating Advice You'll Ever Need!" betont die Bedeutung von Respekt und Kommunikation für gesunde Beziehungen. Wichtige Punkte sind:

- Intimität sollte sich natürlich entwickeln, ohne den Partner zu überfordern.
- Sei offen für verschiedene Typen von Menschen, nicht nur für gemeinsame Interessen.
- Konzentriere dich darauf, ehrlich über dein aktuelles Ich zu sein, anstatt über vergangene Beziehungen zu sprechen.
- Bleibe du selbst und passe dein Verhalten oder Aussehen nicht an die Erwartungen anderer an.
- Achte darauf, dass beide Partner aktiv am Gespräch teilnehmen, um gegenseitiges Interesse zu fördern.
- Reflektiere über deine Gründe für eine Beziehung und priorisiere echtes Glück über äußeren Druck.


In [28]:
print("=" * 60)
print("TOOL TEST: summarize_video")
print("=" * 60)
test_id = extract_video_id(TEST_VIDEO_URL)
answer = ask_agent(f"Please summarize video {test_id}", thread_id="smoke_tools", active_video_id=test_id)
print(answer)


TOOL TEST: summarize_video


The video "The Only Dating Advice You'll Ever Need!" emphasizes the following key points:

- **Respect and Communication**: These are crucial for building healthy relationships.
- **Natural Intimacy**: Intimacy should develop naturally without pressure on the first date.
- **Open to Different Partners**: Shared interests aren't the only basis for connection; be open to various types of partners.
- **Honest Communication**: It's important to communicate feelings and experiences honestly during dating.
- **Focus on the Present**: Concentrate on who you are now rather than discussing past relationships or mistakes.
- **Authenticity**: Don't change yourself to meet someone else's expectations; being authentic attracts compatible partners.
- **Balanced Conversation**: Ensure both individuals share and show interest in each other during conversations.


## 14. LangSmith — Latency & Costs

In [29]:
if os.environ.get("LANGSMITH_TRACING") == "true":
    print("✅ LangSmith tracing active (EU)")
    print(f"   Project:  {os.environ.get('LANGSMITH_PROJECT')}")
    print(f"   Endpoint: {os.environ.get('LANGSMITH_ENDPOINT')}")
    print(f"   Dashboard: https://eu.smith.langchain.com")
    print()
    print("All ask_agent() calls are automatically traced.")
else:
    print("ℹ️  LangSmith inactive (no key configured)")


✅ LangSmith tracing active (EU)
   Project:  youtube-qa-bot
   Endpoint: https://eu.api.smith.langchain.com
   Dashboard: https://eu.smith.langchain.com

All ask_agent() calls are automatically traced.


In [30]:
def session_cost_report():
    """Overview of indexed videos and Whisper costs."""
    try:
        all_docs = vectorstore.get()
        if not all_docs or not all_docs.get("metadatas"):
            print("No videos indexed yet.")
            return
        seen = {}
        for meta in all_docs["metadatas"]:
            vid = meta.get("video_id")
            if vid and vid not in seen:
                seen[vid] = meta
        print("=" * 60)
        print("SESSION REPORT")
        print("=" * 60)
        print(f"Videos: {len(seen)} | Total chunks: {len(all_docs.get('ids', []))}")
        print()
        for i, (vid, meta) in enumerate(seen.items(), 1):
            print(f"  {i}. {meta.get('title', '?')[:55]}")
            print(f"     Whisper: {meta.get('whisper_backend')} ({meta.get('whisper_model')}), Lang: {meta.get('language')}")
    except Exception as e:
        print(f"Error: {e}")

session_cost_report()


SESSION REPORT
Videos: 3 | Total chunks: 14

  1. The Only Dating Advice You'll Ever Need!
     Whisper: api (whisper-1), Lang: english
  2. The Art of War’s Greatest Secret (Win Without Fighting)
     Whisper: api (whisper-1), Lang: english
  3. how to *literally* motivate yourself to do anything
     Whisper: api (whisper-1), Lang: english


## 13b. End-to-End Agent Eval

Runs a set of factual questions through the **full agent loop** (tool selection, retrieval, answer generation) and scores each answer with LLM-as-Judge.

Unlike Benchmark 7 which calls the LLM directly, this tests the complete system:
- Does the agent pick the right tool (`search_video_content` vs. `summarize_video`)?
- Does it retrieve relevant chunks?
- Is the final answer faithful to the transcript?

**Before running:** paste your eval questions into `AGENT_EVAL_QUESTIONS` below.  
Copy them from `all_benchmarks.json` → key `benchmark4_llms` → field `per_q` → `question` + `reference`,  
or from the Benchmark Notebook variable `EVAL_QUESTIONS_BY_VIDEO["medium_video"]`.


In [31]:
# ═══════════════════════════════════════════════════════════
# End-to-End Agent Eval
# ═══════════════════════════════════════════════════════════
# 1. Picks the eval video from ChromaDB (auto or manual override).
# 2. Generates 10 factual questions from the YouTube ORIGINAL transcript
#    — the same source judge_agent_answer uses as ground truth.
#    This ensures question, reference answer and judge all share
#    one consistent ground truth.
# 3. Falls back to all_benchmarks.json if generation fails.
# ───────────────────────────────────────────────────────────

import json as _ej, pandas as pd, time, re as _re
from pathlib import Path as _EP

# ── Optional manual override — set to "" to auto-pick ──────
AGENT_EVAL_VIDEO_ID = ""


def _pick_eval_video() -> tuple[str, str] | tuple[None, None]:
    """Return (video_id, title) of the most recently indexed video in ChromaDB."""
    try:
        all_docs = vectorstore.get()
        if not all_docs or not all_docs.get("metadatas"):
            return None, None
        seen = {}
        for meta in all_docs["metadatas"]:
            vid = meta.get("video_id")
            if vid and vid not in seen:
                seen[vid] = meta.get("title", vid)
        if not seen:
            return None, None
        vid = list(seen.keys())[-1]   # last = most recently inserted
        return vid, seen[vid]
    except Exception as e:
        print(f"⚠️  Could not query ChromaDB: {e}")
        return None, None


def _generate_eval_questions(video_id: str, title: str, n: int = 10) -> list:
    """
    Generate n factual eval questions from the YouTube ORIGINAL transcript
    (get_youtube_reference_transcript / cache).  Using the same source as
    judge_agent_answer guarantees that questions, reference answers and the
    judge verdict all share one consistent ground truth.
    Falls back to cached Whisper text only if no YouTube captions exist.
    Returns list[dict] with keys: question, reference, keywords.
    """
    # ── 1. YouTube original transcript (preferred — matches judge ground truth) ──
    transcript = get_youtube_reference_transcript(video_id)          # cache-first
    source_label = "YouTube captions"

    # ── 2. Whisper full text (fallback when captions are unavailable) ──
    if not transcript:
        transcript = get_cached_whisper_text(video_id)
        source_label = "Whisper transcript (captions unavailable)"

    # ── 3. Deduplicated chunk reconstruction (last resort) ──
    if not transcript:
        docs = vectorstore.get(where={"video_id": video_id})
        if not docs or not docs.get("documents"):
            print("⚠️  No transcript available for question generation.")
            return []
        metas, texts = docs["metadatas"], docs["documents"]
        seen_idx: set = set()
        ordered = []
        for meta, text in sorted(zip(metas, texts),
                                 key=lambda x: x[0].get("chunk_index", 0)):
            ci = meta.get("chunk_index", 0)
            if ci not in seen_idx:
                seen_idx.add(ci)
                ordered.append(text)
        transcript = " ".join(ordered)
        source_label = "Whisper chunks (last resort)"

    if not transcript:
        return []

    print(f"   📄 Generating questions from: {source_label} ({len(transcript)} chars)")
    snippet = transcript[:8000]   # generous window for diverse questions

    prompt = (
        "You are building an evaluation set for a RAG-based Q&A chatbot.\n\n"
        f"VIDEO TITLE: {title}\n\n"
        "TRANSCRIPT (ground truth — same source the judge will use):\n"
        f"\"\"\"\n{snippet}\n\"\"\"\n\n"
        f"Generate exactly {n} factual evaluation questions that:\n"
        "  • Are answerable from the transcript above\n"
        "  • Cover a variety of topics and facts from the video\n"
        "  • Range from simple recall to deeper comprehension\n"
        "  • Each has a concise 1-2 sentence reference answer "
        "    copied or closely paraphrased from the transcript\n\n"
        "IMPORTANT: the reference answer must be grounded in this transcript — "
        "the judge will compare chatbot answers against it verbatim.\n\n"
        "Respond with ONLY valid JSON, no markdown, no extra text:\n"
        "[\n"
        "  {\"question\": \"<question>\","
        " \"reference\": \"<1-2 sentence answer from transcript>\","
        " \"keywords\": [\"<kw1>\", \"<kw2>\", \"<kw3>\"]}\n"
        "  ...\n"
        "]"
    )

    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=CONFIG["helper_model"], temperature=0.2)
    try:
        response = llm.invoke(prompt)
        raw = response.content.strip()
        raw = _re.sub(r"^```(?:json)?\n?", "", raw)
        raw = _re.sub(r"\n?```$", "", raw)
        questions = _ej.loads(raw)
        if isinstance(questions, list) and questions:
            print(f"✅ Generated {len(questions)} eval questions for '{title[:50]}'")
            return questions
    except Exception as e:
        print(f"⚠️  Question generation failed: {e}")
    return []


def _load_eval_questions_from_file() -> list:
    """Load eval questions from all_benchmarks.json if present."""
    candidates = [
        _EP("/content/benchmark_results/all_benchmarks.json"),
        _EP("/content/all_benchmarks.json"),
        _EP.cwd() / "benchmark_results" / "all_benchmarks.json",
    ]
    for path in candidates:
        if path.exists():
            try:
                with open(path) as f:
                    data = _ej.load(f)
                seen, questions = set(), []
                for row in data.get("benchmark4_llms", []):
                    for pq in row.get("per_q", []):
                        q = pq.get("question", "")
                        if q and q not in seen:
                            seen.add(q)
                            questions.append({
                                "question":  q,
                                "reference": pq.get("reference", ""),
                                "keywords":  [],
                            })
                if questions:
                    print(f"✅ Loaded {len(questions)} eval questions from {path}")
                    return questions
            except Exception as e:
                print(f"⚠️  Could not read {path}: {e}")
    return []


# ── Resolve eval video ──────────────────────────────────────
if AGENT_EVAL_VIDEO_ID:
    _eval_vid = AGENT_EVAL_VIDEO_ID
    _docs = vectorstore.get(where={"video_id": _eval_vid}, limit=1)
    _eval_title = (
        _docs["metadatas"][0].get("title", _eval_vid)
        if _docs.get("metadatas") else _eval_vid
    )
    print(f"📌 Using manually set video: {_eval_vid} — {_eval_title}")
else:
    _eval_vid, _eval_title = _pick_eval_video()
    if not _eval_vid:
        print("❌ No videos indexed in ChromaDB. Run the ingestion cell first.")
    else:
        print(f"🎬 Auto-selected video: {_eval_vid} — {_eval_title}")

AGENT_EVAL_VIDEO_ID = _eval_vid

# ── Generate eval questions from YouTube original ───────────
AGENT_EVAL_QUESTIONS = []

if _eval_vid:
    _generated_qs = _generate_eval_questions(_eval_vid, _eval_title, n=10)
    if _generated_qs:
        AGENT_EVAL_QUESTIONS = _generated_qs
    else:
        print("⚠️  Generation failed — trying all_benchmarks.json as fallback.")
        _file_qs = _load_eval_questions_from_file()
        if _file_qs:
            AGENT_EVAL_QUESTIONS = _file_qs
        else:
            print("❌ Could not load or generate any eval questions.")

# ── Run eval ────────────────────────────────────────────────
if not AGENT_EVAL_QUESTIONS or not AGENT_EVAL_VIDEO_ID:
    print("No questions or video — cannot run eval.")
else:
    EVAL_THREAD = "agent_eval_run"
    results = []
    print(f"\nRunning {len(AGENT_EVAL_QUESTIONS)} questions on '{_eval_title[:50]}'...\n")

    for i, q in enumerate(AGENT_EVAL_QUESTIONS, 1):
        t0 = time.time()
        try:
            answer = ask_agent(
                q["question"],
                thread_id=EVAL_THREAD,
                active_video_id=AGENT_EVAL_VIDEO_ID,
            )
            elapsed = time.time() - t0
        except Exception as e:
            print(f"  Q{i}: ❌ Agent error: {e}")
            results.append({
                "q": i, "question": q["question"][:60],
                "answer": f"ERROR: {e}", "score": None,
                "verdict": "error", "latency_sec": 0,
            })
            continue

        try:
            verdict = judge_agent_answer(q["question"], answer, AGENT_EVAL_VIDEO_ID)
            score   = verdict.get("faithfulness_score")
            verd    = verdict.get("verdict", "?")
        except Exception as e:
            score, verd = None, f"judge_error: {e}"

        results.append({
            "q":           i,
            "question":    q["question"][:60],
            "answer":      answer[:120],
            "score":       score,
            "verdict":     verd,
            "latency_sec": round(elapsed, 2),
        })
        score_str = f"{score}/5" if score is not None else "?"
        print(f"  Q{i}: score={score_str}  latency={elapsed:.1f}s  verdict={verd}")

    df_eval = pd.DataFrame(results)
    valid   = df_eval["score"].dropna()
    print(f"\n{'='*60}")
    print(f"AGENT EVAL SUMMARY — {len(AGENT_EVAL_QUESTIONS)} questions on '{_eval_title[:40]}'")
    print(f"{'='*60}")
    print(f"  Avg score:         {valid.mean():.2f}/5" if len(valid) else "  Avg score: n/a")
    print(f"  Min / Max:         {valid.min():.0f} / {valid.max():.0f}" if len(valid) else "")
    print(f"  Fully supported:   {(df_eval['verdict'] == 'fully_supported').sum()}")
    print(f"  Mostly supported:  {(df_eval['verdict'] == 'mostly_supported').sum()}")
    print(f"  Partially:         {(df_eval['verdict'] == 'partially_supported').sum()}")
    print(f"  Unsupported:       {(df_eval['verdict'] == 'unsupported').sum()}")
    print(f"  Avg latency:       {df_eval['latency_sec'].mean():.1f}s")
    print()
    display(df_eval[["q", "question", "score", "verdict", "latency_sec"]])


🎬 Auto-selected video: ubjW4aq_MAg — how to *literally* motivate yourself to do anything
[Captions] ✅ Loaded from cache (16949 chars)
   📄 Generating questions from: YouTube captions (16949 chars)


✅ Generated 10 eval questions for 'how to *literally* motivate yourself to do anythin'

Running 10 questions on 'how to *literally* motivate yourself to do anythin'...



[Captions] ✅ Loaded from cache (16949 chars)


  Q1: score=5/5  latency=6.9s  verdict=fully_supported


[Captions] ✅ Loaded from cache (16949 chars)


...

[Captions] ✅ Loaded from cache (16949 chars)


  Q9: score=5/5  latency=6.9s  verdict=fully_supported


[Captions] ✅ Loaded from cache (16949 chars)


  Q10: score=5/5  latency=3.1s  verdict=fully_supported

AGENT EVAL SUMMARY — 10 questions on 'how to *literally* motivate yourself to '
  Avg score:         5.00/5
  Min / Max:         5 / 5
  Fully supported:   10
  Mostly supported:  0
  Partially:         0
  Unsupported:       0
  Avg latency:       5.7s



,q,question,score,verdict,latency_sec
0,1,What is the main technique suggested for motiv...,5,fully_supported,6.89
1,2,Why does the brain respond better to questions...,5,fully_supported,4.13
2,3,What is interrogative self-talk?,5,fully_supported,3.22
3,4,What are the four reasons why interrogative se...,5,fully_supported,4.78
4,5,How does asking questions affect psychological...,5,fully_supported,6.61
5,6,What happens when you use declarative self-talk?,5,fully_supported,9.03
6,7,What is the importance of autonomy in motivation?,5,fully_supported,6.91
7,8,What does asking yourself questions activate i...,5,fully_supported,5.19
8,9,What is the effect of identity threat on human...,5,fully_supported,6.93
9,10,What did the 2010 study find about interrogati...,5,fully_supported,3.13


## 15. Gradio Web App

Full UI with all features:

- 🎙️ **Whisper** — OpenAI API or local (tiny / base / small / medium / large-v3)
- 🤖 **Agent models** — gpt-4o-mini, gpt-5-mini, gpt-5, gpt-4o, Llama, Qwen
- 📹 **Two video URL inputs** with indexed-video dropdowns
- ⚡ **Compare** — side-by-side 100-char summaries of two indexed videos
- 💬 **Chat with memory** — Thinking... indicator, passages with timestamp links inline
- 🎤 **Microphone input** — transcribed question shown in chat
- 💰 **Slim cost panel** — current request + session totals
- ✅ **Judgement Day tab** — WER check + LLM-as-Judge faithfulness eval
- 🏆 **Benchmark tab** — sweet-spot cards + sortable combo table


In [ ]:
import gradio as gr
from youtube_transcript_api import YouTubeTranscriptApi as _YTA
print("YTA methods:", [m for m in dir(_YTA) if not m.startswith('_')])
import uuid
import time
import orjson as _orjson
import json as _stdjson

# Monkey-patch orjson to handle surrogate chars and emoji > U+FFFF
# that crash Gradio's toorjson serializer
class _SafeOrjson:
    OPT_NON_STR_KEYS = getattr(_orjson, 'OPT_NON_STR_KEYS', 4)
    OPT_SERIALIZE_NUMPY = getattr(_orjson, 'OPT_SERIALIZE_NUMPY', 16)
    OPT_INDENT_2 = getattr(_orjson, 'OPT_INDENT_2', 1)

    @staticmethod
    def dumps(obj, default=None, option=None):
        try:
            return _orjson.dumps(obj, default=default, option=option)
        except TypeError:
            # Fall back to standard json with surrogatepass
            s = _stdjson.dumps(obj, ensure_ascii=False, default=str)
            s = s.encode('utf-8', 'replace')
            return s

    @staticmethod
    def loads(s):
        return _orjson.loads(s)

import gradio.routes as _gr_routes
import types as _types

def _safe_render(content):
    try:
        return _SafeOrjson.dumps(content).decode('utf-8', 'replace')
    except Exception:
        return _stdjson.dumps(content, ensure_ascii=False, default=str)

def _safe_render_str(content):
    return _safe_render(content)

# Patch the ORJSONResponse methods used by Gradio
if hasattr(_gr_routes, 'ORJSONResponse'):
    _gr_routes.ORJSONResponse._render = staticmethod(lambda c: _SafeOrjson.dumps(c))
    _gr_routes.ORJSONResponse._render_str = staticmethod(_safe_render_str)


def _safe(text: str) -> str:
    """Remove surrogate characters that crash orjson/Gradio serialization."""
    if not isinstance(text, str):
        return str(text)
    return text.encode("utf-8", "replace").decode("utf-8")

def _build_model_prices():
    result = {}
    for model, info in BENCHMARK_DATA["llm_prices"].items():
        result[model] = {
            "input":  float(info.get("input", 0.0)),
            "output": float(info.get("output", 0.0)),
        }
    return result

def _build_whisper_prices():
    result = {}
    for model, info in BENCHMARK_DATA["whisper_prices"].items():
        result[model] = float(info.get("cost_per_minute_usd", 0.0))
    return result

MODEL_PRICES = _build_model_prices()
WHISPER_PRICES = _build_whisper_prices()


def _get_available_agent_models_for_ui():
    all_models = CONFIG["available_agent_models"]
    if HF_TOKEN:
        return all_models
    return [m for m in all_models if not m.startswith("hf:")]


AVAILABLE_AGENT_MODELS_UI = _get_available_agent_models_for_ui()


# ═══════════════════════════════════════════════════════════════════
# Session-weite Tracker (Cost + Performance)
# ═══════════════════════════════════════════════════════════════════
SESSION_COSTS = {
    "total_usd": 0.0,
    "agent_usd": 0.0,
    "whisper_usd": 0.0,
    "embedding_usd": 0.0,
    "request_count": 0,
    "last_request_usd": 0.0,
}

SESSION_PERF = {
    "last_combined_sec": 0.0,
    "total_combined_sec": 0.0,
}


def estimate_request_cost(model_name: str, input_tokens: int = 0, output_tokens: int = 0) -> float:
    if model_name not in MODEL_PRICES:
        return 0.0
    prices = MODEL_PRICES[model_name]
    return (
        (input_tokens / 1_000_000) * prices["input"] +
        (output_tokens / 1_000_000) * prices["output"]
    )


def register_agent_call(model_name: str, input_tokens: int, output_tokens: int,
                         elapsed_sec: float = 0.0, whisper_sec: float = 0.0,
                         real_cost_usd: float = None):
    if real_cost_usd is not None:
        cost = real_cost_usd
    else:
        cost = estimate_request_cost(model_name, input_tokens, output_tokens)
    SESSION_COSTS["agent_usd"] += cost
    SESSION_COSTS["total_usd"] += cost
    SESSION_COSTS["request_count"] += 1
    SESSION_COSTS["last_request_usd"] = cost
    combined = elapsed_sec + whisper_sec
    SESSION_PERF["last_combined_sec"] = combined
    SESSION_PERF["total_combined_sec"] += combined


def register_ingest(usd: float, elapsed_sec: float = 0.0, embedding_usd: float = 0.0):
    SESSION_COSTS["whisper_usd"] += usd
    SESSION_COSTS["embedding_usd"] += embedding_usd
    SESSION_COSTS["total_usd"] += usd + embedding_usd
    SESSION_PERF["last_combined_sec"] = elapsed_sec
    SESSION_PERF["total_combined_sec"] += elapsed_sec


def sync_embedding_session_cost():
    global SESSION_EMBEDDING_COST_USD
    already_counted = SESSION_COSTS["embedding_usd"]
    delta = SESSION_EMBEDDING_COST_USD - already_counted
    if delta > 0:
        SESSION_COSTS["embedding_usd"] += delta
        SESSION_COSTS["total_usd"] += delta


UI_STRINGS = {
    "🇬🇧 English": {
        "subtitle":         "Add a video → ask questions (text or voice) → answers with timestamp passages",
        "lang_label":       "🌍 Language",
        "settings_header":  "### ⚙️ Settings",
        "whisper_label":    "🎙️ Whisper model",
        "whisper_info":     "Whisper models accurately converts the video's speech into text, allowing the AI to process the content and answer your questions.",
        "agent_label":      "🤖 Agent model",
        "url_label":        "YouTube URL",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+add",
        "chat_header":      "### 💬 Chat",
        "chat_label":       "Conversation",
        "question_label":   "Question",
        "question_placeholder": "Ask something about the indexed videos...",
        "send_btn":         "📤 Send",
        "compare_missing":  "Please select a second video to compare.",
        "cost_main_header": "### 💰 Cost estimates",
        "cost_last_req":    "Current request cost",
        "cost_last_time":   "Current request time",
        "cost_total":       "Total cost",
        "cost_total_time":  "Total time",
        "eval_header":      "### Ground-Truth Check",
        "eval_desc":        "Compare Whisper output and agent answers against official YouTube captions.",
        "eval_select":      "🎬 Select a video",
        "eval_refresh":     "🔄 Refresh video list",
        "eval_wer_header":  "#### 📊 Transcript quality (WER)",
        "eval_wer_desc":    "*Whisper output vs. YouTube captions.*",
        "eval_wer_btn":     "Check Whisper vs. Youtube transcript",
        "eval_judge_header":"#### ⚖️ Answer quality (Faithfulness)",
        "eval_judge_desc":  "*LLM-as-Judge reviews the last agent answer.*",
        "eval_judge_btn":   "Judge the latest answer",
    },
    "🇩🇪 Deutsch": {
        "subtitle":         "Video einfügen → Fragen stellen (Text oder Sprache) → Antworten mit Timestamp-Passagen",
        "lang_label":       "🌍 Sprache",
        "settings_header":  "### ⚙️ Einstellungen",
        "whisper_label":    "🎙️ Whisper-Modell",
        "whisper_info":     "Whisper wandelt die Sprache des Videos präzise in Text um, damit die KI den Inhalt verarbeiten und Ihre Fragen beantworten kann.",
        "agent_label":      "🤖 Agent-Modell",
        "url_label":        "YouTube URL",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+add",
        "chat_header":      "### 💬 Chat",
        "chat_label":       "Konversation",
        "question_label":   "Frage",
        "question_placeholder": "Frag etwas zu den indizierten Videos...",
        "send_btn":         "📤 Senden",
        "compare_missing":  "Bitte wähle ein zweites Video zum Vergleichen aus.",
        "cost_main_header": "### 💰 Kostenübersicht",
        "cost_last_req":    "Aktuelle Anfrage (Kosten)",
        "cost_last_time":   "Aktuelle Anfrage (Zeit)",
        "cost_total":       "Gesamtkosten",
        "cost_total_time":  "Gesamtzeit",
        "eval_header":      "### Ground-Truth-Check",
        "eval_desc":        "Vergleiche Whisper-Output und Agent-Antworten mit offiziellen YouTube-Captions.",
        "eval_select":      "🎬 Video auswählen",
        "eval_refresh":     "🔄 Video-Liste aktualisieren",
        "eval_wer_header":  "#### 📊 Transkript-Qualität (WER)",
        "eval_wer_desc":    "*Whisper-Output vs. YouTube-Captions.*",
        "eval_wer_btn":     "Whisper vs. Captions prüfen",
        "eval_judge_header":"#### ⚖️ Antwort-Qualität (Faithfulness)",
        "eval_judge_desc":  "*LLM-as-Judge prüft die letzte Agent-Antwort.*",
        "eval_judge_btn":   "Letzte Antwort bewerten",
    },
    "🇪🇸 Español": {
        "subtitle":         "Añade un vídeo → haz preguntas (texto o voz) → respuestas con pasajes y marcas de tiempo",
        "lang_label":       "🌍 Idioma",
        "settings_header":  "### ⚙️ Ajustes",
        "whisper_label":    "🎙️ Modelo Whisper",
        "whisper_info":     "Whisper convierte con precisión el habla del vídeo en texto, permitiendo a la IA procesar el contenido y responder tus preguntas.",
        "agent_label":      "🤖 Modelo del agente",
        "url_label":        "URL de YouTube",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+add",
        "chat_header":      "### 💬 Chat",
        "chat_label":       "Conversación",
        "question_label":   "Pregunta",
        "question_placeholder": "Pregunta algo sobre los vídeos indexados...",
        "send_btn":         "📤 Enviar",
        "compare_missing":  "Por favor selecciona un segundo vídeo para comparar.",
        "cost_main_header": "### 💰 Estimación de costes",
        "cost_last_req":    "Petición actual (coste)",
        "cost_last_time":   "Petición actual (tiempo)",
        "cost_total":       "Coste total",
        "cost_total_time":  "Tiempo total",
        "eval_header":      "### Comprobación Ground-Truth",
        "eval_desc":        "Compara la salida de Whisper y las respuestas del agente con los subtítulos oficiales de YouTube.",
        "eval_select":      "🎬 Selecciona un vídeo",
        "eval_refresh":     "🔄 Actualizar lista",
        "eval_wer_header":  "#### 📊 Calidad del transcripto (WER)",
        "eval_wer_desc":    "*Whisper vs. subtítulos de YouTube.*",
        "eval_wer_btn":     "Comprobar Whisper vs. subtítulos",
        "eval_judge_header":"#### ⚖️ Calidad de la respuesta (Fidelidad)",
        "eval_judge_desc":  "*LLM-as-Judge evalúa la última respuesta.*",
        "eval_judge_btn":   "Evaluar última respuesta",
    },
    "🇫🇷 Français": {
        "subtitle":         "Ajoute une vidéo → pose des questions (texte ou voix) → réponses avec passages horodatés",
        "lang_label":       "🌍 Langue",
        "settings_header":  "### ⚙️ Paramètres",
        "whisper_label":    "🎙️ Modèle Whisper",
        "whisper_info":     "Whisper convertit précisément la parole de la vidéo en texte, permettant à l'IA de traiter le contenu et de répondre à vos questions.",
        "agent_label":      "🤖 Modèle de l'agent",
        "url_label":        "URL YouTube",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+add",
        "chat_header":      "### 💬 Chat",
        "chat_label":       "Conversation",
        "question_label":   "Question",
        "question_placeholder": "Pose une question sur les vidéos indexées...",
        "send_btn":         "📤 Envoyer",
        "compare_missing":  "Veuillez sélectionner une deuxième vidéo pour comparer.",
        "cost_main_header": "### 💰 Estimation des coûts",
        "cost_last_req":    "Requête actuelle (coût)",
        "cost_last_time":   "Requête actuelle (temps)",
        "cost_total":       "Coût total",
        "cost_total_time":  "Temps total",
        "eval_header":      "### Vérification Ground-Truth",
        "eval_desc":        "Compare la sortie de Whisper et les réponses de l'agent aux sous-titres officiels de YouTube.",
        "eval_select":      "🎬 Sélectionner une vidéo",
        "eval_refresh":     "🔄 Actualiser la liste",
        "eval_wer_header":  "#### 📊 Qualité de la transcription (WER)",
        "eval_wer_desc":    "*Whisper vs. sous-titres YouTube.*",
        "eval_wer_btn":     "Vérifier Whisper vs. sous-titres",
        "eval_judge_header":"#### ⚖️ Qualité de la réponse (Fidélité)",
        "eval_judge_desc":  "*LLM-as-Judge évalue la dernière réponse.*",
        "eval_judge_btn":   "Évaluer la dernière réponse",
    },
    "🇹🇷 Türkçe": {
        "subtitle":         "Video ekle → soru sor (metin veya sesli) → zaman damgalı pasajlarla yanıtlar",
        "lang_label":       "🌍 Dil",
        "settings_header":  "### ⚙️ Ayarlar",
        "whisper_label":    "🎙️ Whisper modeli",
        "whisper_info":     "Whisper, videonun konuşmasını hassas bir şekilde metne dönüştürerek yapay zekanın içeriği işlemesini ve sorularınızı yanıtlamasını sağlar.",
        "agent_label":      "🤖 Temsilci modeli",
        "url_label":        "YouTube URL'si",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+",
        "chat_header":      "### 💬 Sohbet",
        "chat_label":       "Konuşma",
        "question_label":   "Soru",
        "question_placeholder": "Dizinlenmiş videolar hakkında bir şeyler sor...",
        "send_btn":         "📤 Gönder",
        "compare_missing":  "Karşılaştırmak için lütfen ikinci bir video seçin.",
        "cost_main_header": "### 💰 Maliyet tahminleri",
        "cost_last_req":    "Güncel istek maliyeti",
        "cost_last_time":   "Güncel istek süresi",
        "cost_total":       "Toplam maliyet",
        "cost_total_time":  "Toplam süre",
        "eval_header":      "### Doğruluk Kontrolü",
        "eval_desc":        "Whisper çıktısını ve temsilci yanıtlarını resmi YouTube altyazılarıyla karşılaştırın.",
        "eval_select":      "🎬 Bir video seçin",
        "eval_refresh":     "🔄 Video listesini yenile",
        "eval_wer_header":  "#### 📊 Deşifre kalitesi (WER)",
        "eval_wer_desc":    "*Whisper çıktısı vs. YouTube altyazıları.*",
        "eval_wer_btn":     "Whisper vs. altyazıları kontrol et",
        "eval_judge_header":"#### ⚖️ Yanıt kalitesi (Doğruluk)",
        "eval_judge_desc":  "*LLM-hakemi, temsilcinin son yanıtını gözden geçirir.*",
        "eval_judge_btn":   "Son yanıtı değerlendir",
    },
    "🇨🇳 中文": {
        "subtitle":         "添加视频 → 提问 (文本或语音) → 带时间戳段落的回答",
        "lang_label":       "🌍 语言",
        "settings_header":  "### ⚙️ 设置",
        "whisper_label":    "🎙️ Whisper 模型",
        "whisper_info":     "Whisper 将视频中的语音精确转换为文本，使 AI 能够处理内容并回答您的问题。",
        "agent_label":      "🤖 代理模型",
        "url_label":        "YouTube 网址",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+添加",
        "chat_header":      "### 💬 聊天",
        "chat_label":       "对话",
        "question_label":   "问题",
        "question_placeholder": "询问有关已索引视频的问题...",
        "send_btn":         "📤 发送",
        "compare_missing":  "请选择第二个视频进行比较。",
        "cost_main_header": "### 💰 成本估算",
        "cost_last_req":    "当前请求成本",
        "cost_last_time":   "当前请求时间",
        "cost_total":       "总成本",
        "cost_total_time":  "总时间",
        "eval_header":      "### 基本事实检查",
        "eval_desc":        "将 Whisper 输出和代理答案与 YouTube 官方字幕进行比较。",
        "eval_select":      "🎬 选择视频",
        "eval_refresh":     "🔄 刷新视频列表",
        "eval_wer_header":  "#### 📊 转录质量 (WER)",
        "eval_wer_desc":    "*Whisper 输出 vs. YouTube 字幕。*",
        "eval_wer_btn":     "检查 Whisper vs. 字幕",
        "eval_judge_header":"#### ⚖️ 答案质量 (忠实度)",
        "eval_judge_desc":  "*LLM 作为法官审查代理的最后答案。*",
        "eval_judge_btn":   "评估最后答案",
    },
    "🇮🇳 हिन्दी": {
        "subtitle":         "एक वीडियो जोड़ें → प्रश्न पूछें (पाठ या आवाज) → टाइमस्टैम्प पैसेज के साथ उत्तर",
        "lang_label":       "🌍 भाषा",
        "settings_header":  "### ⚙️ सेटिंग्स",
        "whisper_label":    "🎙️ व्हिस्पर मॉडल",
        "whisper_info":     "Whisper वीडियो की वाणी को सटीक रूप से टेक्स्ट में परिवर्तित करता है, जिससे AI सामग्री को संसाधित कर आपके प्रश्नों का उत्तर दे सके।",
        "agent_label":      "🤖 एजेंट मॉडल",
        "url_label":        "YouTube URL",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+जोड़ें",
        "chat_header":      "### 💬 चैट",
        "chat_label":       "बातचीत",
        "question_label":   "प्रश्न",
        "question_placeholder": "अनुक्रमित वीडियो के बारे में कुछ पूछें...",
        "send_btn":         "📤 भेजें",
        "compare_missing":  "कृपया तुलना के लिए दूसरा वीडियो चुनें।",
        "cost_main_header": "### 💰 लागत अनुमान",
        "cost_last_req":    "वर्तमान अनुरोध लागत",
        "cost_last_time":   "वर्तमान अनुरोध समय",
        "cost_total":       "कुल लागत",
        "cost_total_time":  "कुल समय",
        "eval_header":      "### ग्राउंड-ट्रुथ जांच",
        "eval_desc":        "व्हिस्पर आउटपुट और एजेंट के उत्तरों की आधिकारिक YouTube कैप्शन के विरुद्ध तुलना करें।",
        "eval_select":      "🎬 एक वीडियो चुनें",
        "eval_refresh":     "🔄 वीडियो सूची ताज़ा करें",
        "eval_wer_header":  "#### 📊 प्रतिलेख गुणवत्ता (WER)",
        "eval_wer_desc":    "*व्हिस्पर आउटपुट बनाम YouTube कैप्शन।*",
        "eval_wer_btn":     "व्हिस्पर बनाम कैप्शन की जांच करें",
        "eval_judge_header":"#### ⚖️ उत्तर गुणवत्ता (सत्यनिष्ठा)",
        "eval_judge_desc":  "*एलएलएम-एक-न्यायाधीश एजेंट के अंतिम उत्तर की समीक्षा करता है।*",
        "eval_judge_btn":   "अंतिम उत्तर का मूल्यांकन करें",
    },
    "🇮🇹 Italiano": {
        "subtitle":         "Aggiungi un video → fai domande (testo o voce) → risposte con passaggi e timestamp",
        "lang_label":       "🌍 Lingua",
        "settings_header":  "### ⚙️ Impostazioni",
        "whisper_label":    "🎙️ Modello Whisper",
        "whisper_info":     "Whisper converte con precisione il parlato del video in testo, consentendo all'IA di elaborare il contenuto e rispondere alle tue domande.",
        "agent_label":      "🤖 Modello dell'agente",
        "url_label":        "URL YouTube",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+add",
        "chat_header":      "### 💬 Chat",
        "chat_label":       "Conversazione",
        "question_label":   "Domanda",
        "question_placeholder": "Chiedi qualcosa sui video indicizzati...",
        "send_btn":         "📤 Invia",
        "compare_missing":  "Seleziona un secondo video per confrontare.",
        "cost_main_header": "### 💰 Stima dei costi",
        "cost_last_req":    "Richiesta attuale (costo)",
        "cost_last_time":   "Richiesta attuale (tempo)",
        "cost_total":       "Costo totale",
        "cost_total_time":  "Tempo totale",
        "eval_header":      "### Controllo Ground-Truth",
        "eval_desc":        "Confronta l'output di Whisper e le risposte dell'agente con i sottotitoli ufficiali di YouTube.",
        "eval_select":      "🎬 Seleziona un video",
        "eval_refresh":     "🔄 Aggiorna elenco",
        "eval_wer_header":  "#### 📊 Qualità del trascritto (WER)",
        "eval_wer_desc":    "*Whisper vs. sottotitoli YouTube.*",
        "eval_wer_btn":     "Controlla Whisper vs. sottotitoli",
        "eval_judge_header":"#### ⚖️ Qualità della risposta (Fedeltà)",
        "eval_judge_desc":  "*LLM-as-Judge valuta l'ultima risposta.*",
        "eval_judge_btn":   "Valuta ultima risposta",
    },
    "🇵🇹 Português": {
        "subtitle":         "Adiciona um vídeo → faz perguntas (texto ou voz) → respostas com passagens e marcas de tempo",
        "lang_label":       "🌍 Idioma",
        "settings_header":  "### ⚙️ Definições",
        "whisper_label":    "🎙️ Modelo Whisper",
        "whisper_info":     "O Whisper converte com precisão o discurso do vídeo em texto, permitindo que a IA processe o conteúdo e responda às suas perguntas.",
        "agent_label":      "🤖 Modelo do agente",
        "url_label":        "URL do YouTube",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+add",
        "chat_header":      "### 💬 Chat",
        "chat_label":       "Conversa",
        "question_label":   "Pergunta",
        "question_placeholder": "Pergunta algo sobre os vídeos indexados...",
        "send_btn":         "📤 Enviar",
        "compare_missing":  "Por favor seleciona um segundo vídeo para comparar.",
        "cost_main_header": "### 💰 Estimativa de custos",
        "cost_last_req":    "Pedido atual (custo)",
        "cost_last_time":   "Pedido atual (tempo)",
        "cost_total":       "Custo total",
        "cost_total_time":  "Tempo total",
        "eval_header":      "### Verificação Ground-Truth",
        "eval_desc":        "Compara a saída do Whisper e as respostas do agente com as legendas oficiais do YouTube.",
        "eval_select":      "🎬 Seleciona um vídeo",
        "eval_refresh":     "🔄 Atualizar lista",
        "eval_wer_header":  "#### 📊 Qualidade da transcrição (WER)",
        "eval_wer_desc":    "*Whisper vs. legendas YouTube.*",
        "eval_wer_btn":     "Verificar Whisper vs. legendas",
        "eval_judge_header":"#### ⚖️ Qualidade da resposta (Fidelidade)",
        "eval_judge_desc":  "*LLM-as-Judge avalia a última resposta.*",
        "eval_judge_btn":   "Avaliar última resposta",
    },
    "🇸🇦 العربية": {
        "subtitle":         "أضف فيديو → اطرح أسئلة (نص أو صوت) → إجابات مع مقاطع ذات طوابع زمنية",
        "lang_label":       "🌍 اللغة",
        "settings_header":  "### ⚙️ الإعدادات",
        "whisper_label":    "🎙️ نموذج ويسبر",
        "whisper_info":     "يحوّل Whisper كلام الفيديو إلى نص بدقة عالية، مما يتيح للذكاء الاصطناعي معالجة المحتوى والإجابة على أسئلتك.",
        "agent_label":      "🤖 نموذج الوكيل",
        "url_label":        "رابط يوتيوب",
        "url_placeholder":  "https://www.youtube.com/watch?v=...",
        "add_btn":          "+إضافة",
        "chat_header":      "### 💬 الدردشة",
        "chat_label":       "المحادثة",
        "question_label":   "سؤال",
        "question_placeholder": "اطرح سؤالاً حول مقاطع الفيديو المفهرسة...",
        "send_btn":         "📤 إرسال",
        "compare_missing":  "يرجى اختيار فيديو ثانٍ للمقارنة.",
        "cost_main_header": "### 💰 تقديرات التكلفة",
        "cost_last_req":    "تكلفة الطلب الحالي",
        "cost_last_time":   "وقت الطلب الحالي",
        "cost_total":       "التكلفة الإجمالية",
        "cost_total_time":  "الوقت الإجمالي",
        "eval_header":      "### التحقق من الحقيقة الأساسية",
        "eval_desc":        "قارن مخرجات ويسبر وإجابات الوكيل مع ترجمات يوتيوب الرسمية.",
        "eval_select":      "🎬 اختر فيديو",
        "eval_refresh":     "🔄 تحديث قائمة الفيديو",
        "eval_wer_header":  "#### 📊 جودة النص (WER)",
        "eval_wer_desc":    "*مخرجات ويسبر مقابل ترجمات يوتيوب.*",
        "eval_wer_btn":     "تحقق من ويسبر مقابل الترجمات",
        "eval_judge_header":"#### ⚖️ جودة الإجابة (الإخلاص)",
        "eval_judge_desc":  "*النموذج اللغوي كحكم يراجع آخر إجابة للوكيل.*",
        "eval_judge_btn":   "تقييم آخر إجابة",
    },
}

DEFAULT_UI_LANG = "🇬🇧 English"


def get_str(lang: str, key: str) -> str:
    return UI_STRINGS.get(lang, UI_STRINGS[DEFAULT_UI_LANG]).get(
        key, UI_STRINGS[DEFAULT_UI_LANG].get(key, "")
    )


# ═══════════════════════════════════════════════════════════════════
# Helper functions
# ═══════════════════════════════════════════════════════════════════

def apply_whisper_choice(choice: str):
    if choice.startswith("api"):
        CONFIG["whisper_backend"] = "api"
    elif choice.startswith("local:"):
        size = choice.replace("local:", "").strip()
        CONFIG["whisper_backend"] = "local"
        CONFIG["whisper_local_size"] = size


def apply_model_choice(model_name: str):
    CONFIG["agent_model"] = model_name
    build_agent(model_name)


def get_video_dropdown_choices() -> list:
    try:
        all_docs = vectorstore.get()
        if not all_docs or not all_docs.get("metadatas"):
            return []
        seen = {}
        for meta in all_docs["metadatas"]:
            vid = meta.get("video_id")
            if vid and vid not in seen:
                seen[vid] = meta.get("title", vid)[:55]
        return [f"{vid} — {title}" for vid, title in seen.items()]
    except Exception:
        return []


def format_cost_panel(lang: str, agent_model: str, whisper_choice: str) -> str:
    # Slim cost panel: only current request + session totals, no per-component breakdown
    return (
        f"- {get_str(lang, 'cost_last_req')}: **${SESSION_COSTS['last_request_usd']:.4f}**\n"
        f"- {get_str(lang, 'cost_last_time')}: **{SESSION_PERF['last_combined_sec']:.2f}s**\n\n"
        f"- {get_str(lang, 'cost_total')}: **${SESSION_COSTS['total_usd']:.4f}**\n"
        f"- {get_str(lang, 'cost_total_time')}: **{SESSION_PERF['total_combined_sec']:.2f}s**\n"
    )


# ═══════════════════════════════════════════════════════════════════
# UI Callbacks
# ═══════════════════════════════════════════════════════════════════

def _refresh_both_dropdowns():
    choices = get_video_dropdown_choices()
    return gr.update(choices=choices), gr.update(choices=choices)


def _make_video_choice_str(video_id: str, title: str) -> str:
    """Build the dropdown value string: 'video_id — title'"""
    return f"{video_id} — {title[:55]}"


def ui_add_video(url: str, whisper_choice: str, lang: str, agent_model: str,
                  progress=gr.Progress()):
    choices = get_video_dropdown_choices()
    dd_no_change = gr.update(choices=choices)

    if not url or not url.strip():
        return ("⚠️  URL?", dd_no_change, dd_no_change, dd_no_change,
                format_cost_panel(lang, agent_model, whisper_choice))
    apply_whisper_choice(whisper_choice)
    try:
        progress(0.2, desc="Downloading audio and transcribing...")
        t0 = time.time()
        result = ingest_video(url.strip())
        elapsed = time.time() - t0

        choices = get_video_dropdown_choices()

        if result["status"] == "already_indexed":
            # Find existing entry for this video
            vid = result["video_id"]
            existing = next((c for c in choices if c.startswith(vid)), choices[-1] if choices else "")
            dd_set = gr.update(choices=choices, value=existing)
            dd_refresh = gr.update(choices=choices)
            return (
                "ℹ️  This video is already indexed.",
                dd_set, dd_refresh, dd_set,
                format_cost_panel(lang, agent_model, whisper_choice),
            )

        whisper_sec = result.get("whisper_time_sec", elapsed)
        register_ingest(
            result.get("whisper_cost_usd", 0.0),
            whisper_sec,
            embedding_usd=result.get("embedding_cost_usd", 0.0),
        )

        new_choice = _make_video_choice_str(result["video_id"], result["title"])
        # Set new video as selected value in dropdown 1 (this button's dropdown)
        dd1_new = gr.update(choices=choices, value=new_choice)
        dd2_refresh = gr.update(choices=choices)
        eval_refresh = gr.update(choices=choices, value=new_choice)  # pre-select in eval tab

        msg = (
            f"✅ **{_safe(result['title'][:40])}**\n"
            f"{_safe(result['channel'][:30])}\n"
            f"{result['num_chunks']} chunks · ${result['whisper_cost_usd']:.4f}"
        )
        return (msg, dd1_new, dd2_refresh, eval_refresh,
                format_cost_panel(lang, agent_model, whisper_choice))
    except Exception as e:
        choices = get_video_dropdown_choices()
        dd_refresh = gr.update(choices=choices)
        return (f"❌ {str(e)[:60]}", dd_refresh, dd_refresh, dd_refresh,
                format_cost_panel(lang, agent_model, whisper_choice))


def ui_add_video_2(url: str, whisper_choice: str, lang: str, agent_model: str,
                    progress=gr.Progress()):
    """Second URL input — sets new video in dropdown 2 instead of dropdown 1."""
    choices_before = get_video_dropdown_choices()
    dd_no_change = gr.update(choices=choices_before)

    if not url or not url.strip():
        return ("⚠️  URL?", dd_no_change, dd_no_change, dd_no_change,
                format_cost_panel(lang, agent_model, whisper_choice))
    apply_whisper_choice(whisper_choice)
    try:
        progress(0.2, desc="Downloading audio and transcribing...")
        t0 = time.time()
        result = ingest_video(url.strip())
        elapsed = time.time() - t0

        choices = get_video_dropdown_choices()

        if result["status"] == "already_indexed":
            vid = result["video_id"]
            existing = next((c for c in choices if c.startswith(vid)), choices[-1] if choices else "")
            dd_set = gr.update(choices=choices, value=existing)
            dd_refresh = gr.update(choices=choices)
            return (
                "ℹ️  This video is already indexed.",
                dd_refresh, dd_set, dd_set,
                format_cost_panel(lang, agent_model, whisper_choice),
            )

        whisper_sec = result.get("whisper_time_sec", elapsed)
        register_ingest(
            result.get("whisper_cost_usd", 0.0),
            whisper_sec,
            embedding_usd=result.get("embedding_cost_usd", 0.0),
        )

        new_choice = _make_video_choice_str(result["video_id"], result["title"])
        dd1_refresh = gr.update(choices=choices)
        # Set new video as selected value in dropdown 2 (this button's dropdown)
        dd2_new = gr.update(choices=choices, value=new_choice)
        eval_refresh = gr.update(choices=choices, value=new_choice)  # pre-select in eval tab

        msg = (
            f"✅ **{_safe(result['title'][:40])}**\n"
            f"{_safe(result['channel'][:30])}\n"
            f"{result['num_chunks']} chunks · ${result['whisper_cost_usd']:.4f}"
        )
        return (msg, dd1_refresh, dd2_new, eval_refresh,
                format_cost_panel(lang, agent_model, whisper_choice))
    except Exception as e:
        choices = get_video_dropdown_choices()
        dd_refresh = gr.update(choices=choices)
        return (f"❌ {str(e)[:60]}", dd_refresh, dd_refresh, dd_refresh,
                format_cost_panel(lang, agent_model, whisper_choice))


_PASSAGE_KEYWORDS = {
    "passage", "passages", "source", "sources", "excerpt", "excerpts",
    "quote", "quotes", "example", "examples", "evidence", "reference",
    "show me", "where", "timestamp", "timestamps", "context",
    "passagen", "quelle", "quellen", "zeig", "belege",
}

def _user_wants_passages(message: str) -> bool:
    msg_lower = message.lower()
    return any(kw in msg_lower for kw in _PASSAGE_KEYWORDS)


def ui_chat(message: str, history: list, session_id: str, model_name: str,
            lang: str, whisper_choice: str, active_video_choice: str = ""):
    """Generator: yields Thinking... first, then real answer (+ passages only if requested)."""
    if not message or not message.strip():
        yield history, session_id, "", format_cost_panel(lang, model_name, whisper_choice), ""
        return

    # Parse active video — done here so agent gets context from the start
    video_id_filter = None
    video_title_filter = None
    if active_video_choice and " — " in active_video_choice:
        parts = active_video_choice.split(" — ", 1)
        video_id_filter = parts[0].strip()
        video_title_filter = parts[1].strip() if len(parts) > 1 else None

    # Step 1: show Thinking...
    thinking_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": "Thinking..."},
    ]
    yield thinking_history, session_id, "", format_cost_panel(lang, model_name, whisper_choice), ""

    # Step 2: run agent — active video passed so agent restricts search to it
    try:
        t0 = time.time()
        agent_result = ask_agent_with_usage(
            message,
            thread_id=session_id,
            model_name=model_name,
            active_video_id=video_id_filter,
            active_video_title=video_title_filter,
        )
        answer = agent_result["answer"]
        elapsed = time.time() - t0
        register_agent_call(
            model_name,
            input_tokens=agent_result["input_tokens"],
            output_tokens=agent_result["output_tokens"],
            elapsed_sec=elapsed,
            real_cost_usd=agent_result["cost_usd"],
        )
        sync_embedding_session_cost()
    except Exception as e:
        answer = f"❌ {e}"

    # Step 3: always fetch top 3 passages for the passages panel (right column)
    # Passages in chat only if user explicitly requested them
    top_passages_md = ""
    try:
        passages = get_top_passages_with_metadata(message, k=3, video_id=video_id_filter)
        if passages:
            top_passages_md = _safe(format_passages_as_markdown(passages, show_timestamps=True))
    except Exception:
        top_passages_md = ""

    passages_in_chat = ""
    if _user_wants_passages(message) and top_passages_md:
        passages_in_chat = top_passages_md

    full_answer = _safe(f"{answer}\n\n---\n{passages_in_chat}" if passages_in_chat else answer)
    final_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": full_answer},
    ]
    yield final_history, session_id, "", format_cost_panel(lang, model_name, whisper_choice), top_passages_md


def ui_audio_to_chat(audio_file, history, session_id, model_name, lang,
                      whisper_choice, active_video_choice=""):
    """Triggered after audio recording is fully uploaded. Transcribes via
    Whisper, then runs the agent.

    Returns 5 values to match the chat handler's outputs:
        chatbot, session_state, audio_input (cleared), cost_panel, passages_output
    """
    cost = format_cost_panel(lang, model_name, whisper_choice)

    # Defensive: no audio at all
    if audio_file is None:
        warn_history = history + [{
            "role": "assistant",
            "content": "🎤 *No audio captured. Try recording again — make sure to grant the browser microphone permission.*",
        }]
        return warn_history, session_id, None, cost, ""

    try:
        # Parse active video — same as ui_chat
        video_id_filter = None
        video_title_filter = None
        if active_video_choice and " — " in active_video_choice:
            parts = active_video_choice.split(" — ", 1)
            video_id_filter = parts[0].strip()
            video_title_filter = parts[1].strip() if len(parts) > 1 else None

        # Transcribe
        t_w = time.time()
        transcribed = transcribe_user_audio(audio_file)
        w_elapsed = time.time() - t_w

        if not transcribed or not transcribed.strip():
            warn_history = history + [{
                "role": "assistant",
                "content": "🎤 *Audio was empty or could not be transcribed. Speak a bit louder/longer and try again.*",
            }]
            return warn_history, session_id, None, cost, ""

        # Run the agent on the transcript
        t_a = time.time()
        agent_result = ask_agent_with_usage(
            transcribed,
            thread_id=session_id,
            model_name=model_name,
            active_video_id=video_id_filter,
            active_video_title=video_title_filter,
        )
        answer = agent_result["answer"]
        a_elapsed = time.time() - t_a

        register_agent_call(
            model_name,
            input_tokens=agent_result["input_tokens"],
            output_tokens=agent_result["output_tokens"],
            elapsed_sec=a_elapsed,
            whisper_sec=w_elapsed,
            real_cost_usd=agent_result["cost_usd"],
        )
        sync_embedding_session_cost()

        # Passages only if explicitly requested
        passages_md = ""
        if _user_wants_passages(transcribed):
            try:
                passages = get_top_passages_with_metadata(
                    transcribed, k=3, video_id=video_id_filter
                )
                if passages:
                    passages_md = _safe(
                        format_passages_as_markdown(passages, show_timestamps=True)
                    )
            except Exception as e:
                passages_md = f"*Retrieval error: {e}*"

        full_answer = _safe(answer)
        final_history = history + [
            {"role": "user", "content": f"🎤 {transcribed}"},
            {"role": "assistant", "content": full_answer},
        ]
        cost = format_cost_panel(lang, model_name, whisper_choice)
        # Clear audio_input (None) so the next recording fires .change() again.
        return final_history, session_id, None, cost, passages_md
    except Exception as e:
        err_history = history + [{
            "role": "assistant",
            "content": f"❌ Audio error: {str(e)[:200]}",
        }]
        return err_history, session_id, None, cost, ""


def ui_compare_whisper_captions(video_choice: str):
    if not video_choice:
        return "⚠️  No video selected. Click **🔄 Refresh** to load already-indexed videos, then select one."
    vid = video_choice.split(" — ")[0]
    result = compare_whisper_vs_captions(vid)
    if result["status"] == "no_captions":
        return f"ℹ️  {result['message']}"
    if result["status"] == "error":
        return f"❌ {result['message']}"
    return (
        f"### 📊 Whisper vs. YouTube Captions\n\n"
        f"**WER: {result['wer_percent']}%**\n\n"
        f"- Whisper: {result['whisper_length']} chars\n"
        f"- Reference: {result['reference_length']} chars\n\n"
        f"**Whisper preview:**\n> {result['whisper_preview']}\n\n"
        f"**Reference preview:**\n> {result['reference_preview']}"
    )


def ui_judge_last_answer(video_choice: str, history: list, active_video: str = ""):
    # Determine which video to judge against:
    # active_video (= what the chatbot is currently using) takes priority.
    # video_choice (eval dropdown) is the fallback if active_video is not set.
    resolved_video = active_video.strip() if active_video and active_video.strip() else video_choice
    if not resolved_video:
        return "⚠️  No video selected. Switch to the **💬 Chat Interface Hub** tab, select a video, then come back here."
    if not history or len(history) < 2:
        return "⚠️  No Q&A found yet. Please ask a question in the **💬 Chat Interface Hub** tab first, then come back here to judge the answer."
    last_user = None
    last_assistant = None
    for msg in reversed(history):
        if msg["role"] == "assistant" and last_assistant is None:
            last_assistant = msg["content"]
        elif msg["role"] == "user" and last_assistant is not None and last_user is None:
            last_user = msg["content"]
            break
    if not last_user or not last_assistant:
        return "⚠️  Could not find a complete Q/A pair."
    # resolved_video may be "video_id — Title" or just "video_id"
    vid = resolved_video.split(" — ")[0].strip()
    result = judge_agent_answer(last_user, last_assistant, vid)
    if result["status"] == "no_captions":
        return f"ℹ️  {result['message']}"
    if result["status"] == "error":
        return f"❌ {result['message']}"
    supported = "\n".join(f"- ✅ {c}" for c in result.get("supported_claims", []))
    unsupported = "\n".join(f"- ⚠️  {c}" for c in result.get("unsupported_claims", []))
    video_label = resolved_video if " — " in resolved_video else vid
    return (
        f"### ⚖️ Faithfulness Judge ({result['judge_model']})\n\n"
        f"🎬 **Video:** `{video_label}`\n\n"
        f"**Q:** {last_user}\n\n"
        f"**A:** {last_assistant}\n\n"
        f"**Score: {result['faithfulness_score']}/5** — *{result['verdict']}*\n\n"
        f"{result['explanation']}\n\n"
        f"**Supported:**\n{supported or '*(none)*'}\n\n"
        f"**Unsupported:**\n{unsupported or '*(none)*'}"
    )


def ui_compare_two_videos(choice1: str, choice2: str, lang: str, model_name: str):
    """Compare two indexed videos using their dropdown values.
    Fully isolated from chat state — reads directly from the two dropdowns."""
    def _vid_from_choice(choice):
        if not choice or " — " not in choice:
            return None
        return choice.split(" — ")[0].strip()

    vid1 = _vid_from_choice(choice1)
    vid2 = _vid_from_choice(choice2)

    if not vid1 or not vid2:
        return get_str(lang, "compare_missing")
    if vid1 == vid2:
        return get_str(lang, "compare_missing")

    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=CONFIG["helper_model"], temperature=0)
    results = []
    for vid in [vid1, vid2]:
        full_text, meta = _get_full_transcript(vid)
        if full_text is None:
            results.append(f"**{vid}** — not indexed yet.")
            continue
        snippet = full_text[:3000]
        resp = llm.invoke(
            f"Summarize this video transcript in max 200 characters:\n\n{snippet}"
        )
        title = meta.get("title", vid)[:40]
        results.append(f"**{title}:** {resp.content.strip()[:200]}")

    return "\n\n".join(results)


def change_ui_language(lang_choice: str, agent_model: str, whisper_choice: str):
    s = UI_STRINGS.get(lang_choice, UI_STRINGS[DEFAULT_UI_LANG])
    return (
        s["subtitle"],
        gr.update(label=s["lang_label"]),
        s["settings_header"],
        gr.update(label=s["whisper_label"], info=s["whisper_info"]),
        gr.update(label=s["agent_label"]),
        gr.update(label=s["url_label"], placeholder=s["url_placeholder"]),
        gr.update(value=s["add_btn"]),
        gr.update(label=s["url_label"], placeholder=s["url_placeholder"]),
        gr.update(value=s["add_btn"]),
        s["chat_header"],
        gr.update(label=s["chat_label"]),
        gr.update(label=s["question_label"], placeholder=s["question_placeholder"]),
        gr.update(value=s["send_btn"]),
        s["cost_main_header"],
        format_cost_panel(lang_choice, agent_model, whisper_choice),
        s["eval_header"],
        s["eval_desc"],
        gr.update(label=s["eval_select"]),
        gr.update(value=s["eval_refresh"]),
        s["eval_wer_header"],
        s["eval_wer_desc"],
        gr.update(value=s["eval_wer_btn"]),
        s["eval_judge_header"],
        s["eval_judge_desc"],
        gr.update(value=s["eval_judge_btn"]),
    )


# ═══════════════════════════════════════════════════════════════════
# Theme + CSS
# ═══════════════════════════════════════════════════════════════════

def _get_last_indexed_video() -> str:
    """Return the dropdown string of the last indexed video, or empty string."""
    try:
        choices = get_video_dropdown_choices()
        return choices[-1] if choices else ""
    except Exception:
        return ""


DARK_THEME = gr.themes.Soft(
    primary_hue="indigo",
    neutral_hue="slate",
).set(
    body_background_fill="*neutral_950",
    body_background_fill_dark="*neutral_950",
    background_fill_primary="*neutral_900",
    background_fill_primary_dark="*neutral_900",
    background_fill_secondary="*neutral_800",
    background_fill_secondary_dark="*neutral_800",
    border_color_primary="*neutral_700",
    body_text_color="*neutral_100",
    body_text_color_dark="*neutral_100",
)

BOX_STYLE = """
background: linear-gradient(135deg, #1e293b 0%, #1a2540 100%) !important;
border: 1px solid #334155 !important;
border-radius: 12px !important;
padding: 16px !important;
"""

CUSTOM_CSS = f"""
.gradio-container {{ max-width: 100% !important; background: linear-gradient(160deg, #0a0f1e 0%, #0f172a 40%, #1e1b4b 70%, #0f172a 100%) !important; }}
footer {{ display: none !important; }}

/* Gradient title */
#app-title h1 {{
    text-align: center !important;
    background: linear-gradient(135deg, #6366f1, #a78bfa, #ec4899);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    font-size: 2.2em !important;
    font-weight: 800 !important;
    letter-spacing: -0.5px;
}}
#app-subtitle p {{ text-align: center !important; color: #94a3b8 !important; }}

#lang-wrap {{ max-width: 25% !important; margin-left: auto !important; margin-right: auto !important; }}

/* Gradient border on boxes */
#add-video-box {{
    margin-top: 12px !important;
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 14px !important;
}}

#cost-box {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #0ea5e9, #6366f1) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 14px !important;
}}

#compare-box {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 14px !important;
    margin-top: 10px !important;
}}

/* Top passages box */
#passages-box {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 14px !important;
    margin-top: 10px !important;
}}

/* Gradient send button */
#send-btn {{
    min-height: 82px !important;
    background: linear-gradient(135deg, #6366f1, #8b5cf6, #ec4899) !important;
    border: none !important;
    font-weight: 700 !important;
}}
#send-btn:hover {{
    background: linear-gradient(135deg, #4f46e5, #7c3aed, #db2777) !important;
    transform: translateY(-1px);
    box-shadow: 0 4px 20px rgba(99, 102, 241, 0.4) !important;
}}

/* Add button gradient */
.add-btn-grad {{
    background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
    border: none !important;
}}

#add-right-col {{ gap: 6px !important; }}
#add-status-compact {{ font-size: 11px !important; line-height: 1.3 !important; }}
#add-status-compact > * {{ margin: 0 !important; padding: 4px 6px !important; }}

#mic-hidden {{ display: none !important; }}
#mic-btn {{
    min-height: 82px !important;
    height: 82px !important;
    font-size: 28px !important;
    padding: 0 !important;
    background: linear-gradient(135deg, #0f172a, #1e293b) !important;
    border: 1px solid #475569 !important;
}}
#mic-btn.recording {{
    background: linear-gradient(135deg, #dc2626, #9f1239) !important;
    animation: pulse 1.2s ease-in-out infinite;
}}
@keyframes pulse {{
    0%, 100% {{ opacity: 1; }}
    50% {{ opacity: 0.6; }}
}}

/* Chatbot gradient border */
.chatbot-wrap {{
    border: 1px solid #334155 !important;
    border-radius: 12px !important;
    overflow: hidden;
}}

/* Tab styling */
.tab-nav button {{
    background: transparent !important;
    border-bottom: 2px solid transparent !important;
    transition: all 0.2s;
}}
.tab-nav button.selected {{
    border-bottom: 2px solid #6366f1 !important;
    color: #a78bfa !important;
}}

/* Winner cards gradient */
.winner-card {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 10px !important;
    padding: 12px !important;
    margin: 4px !important;
}}

/* Settings dropdowns box */
#settings-box {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 14px !important;
    margin-top: 10px !important;
}}

/* Chat input area */
#chat-input-row {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 10px !important;
    margin-top: 8px !important;
}}

/* Compare button */
#compare-btn {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 10px !important;
    margin-top: 8px !important;
}}

/* Judgement Day — video select box */
#eval-select-box {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 14px !important;
    margin-bottom: 10px !important;
}}

/* Header box above tabs */
#header-box {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 16px !important;
    margin-bottom: 12px !important;
}}

/* Judgement Day eval columns */
.eval-box {{
    background: linear-gradient(#1e293b, #1a2540) padding-box,
                linear-gradient(135deg, #6366f1, #ec4899) border-box !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 14px !important;
    margin-top: 10px !important;
}}
"""

MIC_JS = """
() => {
    const ourBtn = document.getElementById('mic-btn');
    if (!ourBtn || ourBtn.dataset.wired) return;
    ourBtn.dataset.wired = '1';

    const findRecordButton = () => {
        const hidden = document.getElementById('mic-hidden');
        if (!hidden) return null;
        const buttons = hidden.querySelectorAll('button');
        for (const b of buttons) {
            const label = (b.getAttribute('aria-label') || '').toLowerCase();
            if (label.includes('record') || label.includes('aufnehm') || label.includes('stop')) {
                return b;
            }
        }
        return buttons[0] || null;
    };

    let recording = false;
    ourBtn.addEventListener('click', (e) => {
        e.preventDefault();
        const recBtn = findRecordButton();
        if (!recBtn) { console.warn('Mic: record button not found'); return; }
        recBtn.click();
        recording = !recording;
        if (recording) {
            ourBtn.classList.add('recording');
            ourBtn.textContent = '\u23F9';
        } else {
            ourBtn.classList.remove('recording');
            ourBtn.textContent = '\uD83C\uDF99\uFE0F';
        }
    });
}
"""


# ═══════════════════════════════════════════════════════════════════
# Benchmark Tab helpers
# ═══════════════════════════════════════════════════════════════════

# Fixed business sweet spot combinations (not from winners.json)
_SWEET_SPOTS = [
    {
        "tier": "Solo Developer",
        "emoji": "👨‍💻",
        "whisper": "local: small",
        "llm": "hf:llama-3.3",
        "note": "Near-zero cost — local Whisper + free HuggingFace LLM",
        "cost_label": "≈ $0.00",
    },
    {
        "tier": "Start-up",
        "emoji": "🚀",
        "whisper": "local: small",
        "llm": "gpt-4o-mini or llama",
        "note": "Low cost — solid transcription, flexible LLM choice",
        "cost_label": "≈ $0.001 / query",
    },
    {
        "tier": "Mid-size Company",
        "emoji": "🏢",
        "whisper": "api (whisper-1)",
        "llm": "gpt-5-mini",
        "note": "Production-ready — API Whisper + latest efficient GPT",
        "cost_label": "≈ $0.01–0.05 / query",
    },
    {
        "tier": "Enterprise",
        "emoji": "🏛",
        "whisper": "api (whisper-1)",
        "llm": "gpt-5-mini or higher",
        "note": "Best quality — OpenAI API end-to-end",
        "cost_label": "≈ $0.05+ / query",
    },
]

def _render_sweet_spot_card(s: dict) -> str:
    return (
        f"**{s['emoji']} {s['tier']}**  \n"
        f"🎙️ `{s['whisper']}`  \n"
        f"🤖 `{s['llm']}`  \n"
        f"💰 {s['cost_label']}  \n"
        f"*{s['note']}*"
    )

# _render_zero_cost_card removed — "Almost Zero Costs Hero" tile dropped

def _get_combo_df():
    import pandas as pd
    ct = BENCHMARK_DATA.get("combo_table")
    if ct is None or (hasattr(ct, "empty") and ct.empty):
        return pd.DataFrame([{"info": "No benchmark data — run benchmark_final.ipynb first"}])
    display_cols = ["whisper_model", "llm_model", "whisper_wer_percent",
                    "llm_avg_score", "combo_total_cost_usd", "judge_thumbs"]
    available = [c for c in display_cols if c in ct.columns]
    df = ct[available].copy()
    # Sort: first by avg score desc, then by cost asc
    if "llm_avg_score" in df.columns and "combo_total_cost_usd" in df.columns:
        df = df.sort_values(
            ["llm_avg_score", "combo_total_cost_usd"],
            ascending=[False, True],
        ).reset_index(drop=True)
    return df


# ═══════════════════════════════════════════════════════════════════
# UI Layout
# ═══════════════════════════════════════════════════════════════════

with gr.Blocks(
    title="YouTube Analyzer",
    theme=DARK_THEME,
    css=CUSTOM_CSS,
    js="() => { document.body.classList.add('dark'); }",
) as demo:
    with gr.Group(elem_id="header-box"):
        gr.Markdown("# 🎥 Multilingual YouTube Analyzer", elem_id="app-title")
        with gr.Row(elem_id="lang-wrap"):
            ui_lang_dropdown = gr.Dropdown(
                choices=list(UI_STRINGS.keys()),
                value=DEFAULT_UI_LANG,
                label=get_str(DEFAULT_UI_LANG, "lang_label"),
                container=True,
            )
        subtitle_md = gr.Markdown(
            get_str(DEFAULT_UI_LANG, "subtitle"),
            elem_id="app-subtitle",
        )

    session_state = gr.State(value=str(uuid.uuid4()))
    active_video_state = gr.State(value=_get_last_indexed_video())

    with gr.Tabs():
        with gr.Tab("💬 Chat Interface Hub"):
            with gr.Row(equal_height=False):

                # ── LEFT ──
                with gr.Column(scale=1):
                    settings_header_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "settings_header"))
                    with gr.Group(elem_id="settings-box"):
                        whisper_dropdown = gr.Dropdown(
                            choices=CONFIG["available_whisper_options"],
                            value=CONFIG["available_whisper_options"][0],
                            label=get_str(DEFAULT_UI_LANG, "whisper_label"),
                            info=get_str(DEFAULT_UI_LANG, "whisper_info"),
                        )
                        model_dropdown = gr.Dropdown(
                            choices=AVAILABLE_AGENT_MODELS_UI,
                            value=CONFIG["agent_model"] if CONFIG["agent_model"] in AVAILABLE_AGENT_MODELS_UI else AVAILABLE_AGENT_MODELS_UI[0],
                            label=get_str(DEFAULT_UI_LANG, "agent_label"),
                        )

                    # ── Video 1 ──
                    with gr.Group(elem_id="add-video-box"):
                        with gr.Row():
                            url_input = gr.Textbox(
                                label=get_str(DEFAULT_UI_LANG, "url_label"),
                                placeholder=get_str(DEFAULT_UI_LANG, "url_placeholder"),
                                scale=4,
                            )
                            add_btn = gr.Button(
                                get_str(DEFAULT_UI_LANG, "add_btn"),
                                variant="primary",
                                scale=1,
                                min_width=70,
                                elem_classes=["add-btn-grad"],
                            )
                        indexed_dropdown_1 = gr.Dropdown(
                            choices=get_video_dropdown_choices(),
                            label="Indexed videos",
                            scale=1,
                        )
                        add_status = gr.Markdown(elem_id="add-status-compact")

                    # ── Video 2 ──
                    with gr.Group(elem_id="add-video-box"):
                        with gr.Row():
                            url_input_2 = gr.Textbox(
                                label=get_str(DEFAULT_UI_LANG, "url_label"),
                                placeholder=get_str(DEFAULT_UI_LANG, "url_placeholder"),
                                scale=4,
                            )
                            add_btn_2 = gr.Button(
                                get_str(DEFAULT_UI_LANG, "add_btn"),
                                variant="primary",
                                scale=1,
                                min_width=70,
                                elem_classes=["add-btn-grad"],
                            )
                        indexed_dropdown_2 = gr.Dropdown(
                            choices=get_video_dropdown_choices(),
                            label="Indexed videos",
                            scale=1,
                        )
                        add_status_2 = gr.Markdown(elem_id="add-status-compact")

                    # ── Compare button ──
                    compare_btn = gr.Button("⚡ Compare", variant="secondary", elem_id="compare-btn")

                # ── MIDDLE ──
                with gr.Column(scale=1):
                    chat_header_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "chat_header"))
                    chatbot = gr.Chatbot(
                        type="messages",
                        height=430,
                        label=get_str(DEFAULT_UI_LANG, "chat_label"),
                        elem_classes=["chatbot-wrap"],
                    )

                    with gr.Group(elem_id="chat-input-row"):
                        with gr.Row(equal_height=True):
                            msg_input = gr.Textbox(
                                label=get_str(DEFAULT_UI_LANG, "question_label"),
                                placeholder=get_str(DEFAULT_UI_LANG, "question_placeholder"),
                                lines=2,
                                scale=6,
                            )
                            mic_btn = gr.Button("🎤", elem_id="mic-btn", scale=1, min_width=70)
                            send_btn = gr.Button(
                                get_str(DEFAULT_UI_LANG, "send_btn"),
                                variant="primary",
                                scale=4,
                                elem_id="send-btn",
                            )

                    audio_input = gr.Audio(
                        sources=["microphone"],
                        type="filepath",
                        label="",
                        show_label=False,
                        elem_id="mic-hidden",
                    )

                # ── RIGHT ──
                with gr.Column(scale=1):
                    cost_header_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "cost_main_header"))
                    cost_panel = gr.Markdown(
                        value=format_cost_panel(
                            DEFAULT_UI_LANG,
                            CONFIG["agent_model"],
                            CONFIG["available_whisper_options"][0],
                        ),
                        elem_id="cost-box",
                    )
                    # ── Compare output ──
                    compare_output = gr.Markdown(elem_id="compare-box")
                    # ── Top 3 passages output ──
                    passages_output = gr.Markdown(elem_id="passages-box")

        with gr.Tab("✅ Judgement Day"):
            eval_header_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "eval_header"))
            eval_desc_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "eval_desc"))

            with gr.Group(elem_id="eval-select-box"):
                video_eval_dropdown = gr.Dropdown(
                    choices=get_video_dropdown_choices(),
                    label=get_str(DEFAULT_UI_LANG, "eval_select"),
                )
                refresh_eval_btn = gr.Button(
                    get_str(DEFAULT_UI_LANG, "eval_refresh"),
                    size="sm",
                )

            with gr.Row():
                with gr.Column():
                    with gr.Group(elem_classes=["eval-box"]):
                        eval_wer_header_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "eval_wer_header"))
                        eval_wer_desc_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "eval_wer_desc"))
                        wer_btn = gr.Button(get_str(DEFAULT_UI_LANG, "eval_wer_btn"), variant="primary")
                        wer_output = gr.Markdown()

                with gr.Column():
                    with gr.Group(elem_classes=["eval-box"]):
                        eval_judge_header_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "eval_judge_header"))
                        eval_judge_desc_md = gr.Markdown(get_str(DEFAULT_UI_LANG, "eval_judge_desc"))
                        judge_btn = gr.Button(get_str(DEFAULT_UI_LANG, "eval_judge_btn"), variant="primary")
                        judge_output = gr.Markdown()

        with gr.Tab("🏆 Benchmark"):
            gr.Markdown("## Benchmark Results — Whisper × LLM Combinations")
            gr.Markdown(
                "Data from benchmark_final.ipynb. Each row is a real end-to-end measurement over 10 eval questions."
            )

            # ── Business Sweet Spots (4 tiles) ──
            with gr.Row():
                for spot in _SWEET_SPOTS:
                    with gr.Column(elem_classes=["winner-card"]):
                        gr.Markdown(_render_sweet_spot_card(spot))

            # ── Combo table ──
            combo_dataframe = gr.Dataframe(
                value=_get_combo_df(),
                label="All Combinations — sorted by score desc, cost asc",
                wrap=True,
                interactive=False,
            )

        with gr.Tab("📊 Benchmark Plots"):
            gr.Markdown("## 📊 Benchmark Plots")
            gr.Markdown(
                "Visual results from all benchmark runs across videos, "
                "Whisper models, LLMs and chunk configurations."
            )

            # Search for benchmark plots in multiple locations.
            # Priority: benchmark_dir from BENCHMARK_DATA, then the new
            # benchmark_imports_for_QandA_notebook folder, then legacy locations.
            def _find_plot(filename):
                import os
                candidates = []
                bdir = BENCHMARK_DATA.get("benchmark_dir") or ""
                if bdir:
                    candidates.append(os.path.join(bdir, filename))
                # Preferred new location
                candidates.append(str(BASE_DIR / "benchmark_imports_for_QandA_notebook" / filename))
                # Backwards-compat (legacy)
                candidates.append(str(BASE_DIR / "benchmark_results" / filename))
                candidates.append(str(BASE_DIR / filename))
                if str(BASE_DIR) != "/content":
                    candidates.append(f"/content/benchmark_imports_for_QandA_notebook/{filename}")
                    candidates.append(f"/content/benchmark_results/{filename}")
                    candidates.append(f"/content/{filename}")
                for path in candidates:
                    if os.path.exists(path):
                        return path
                return None

            def _plot(filename, caption=""):
                path = _find_plot(filename)
                if path:
                    return gr.Image(
                        value=path,
                        label=caption,
                        show_label=bool(caption),
                        interactive=False,
                    )
                _hint_dir = str(BASE_DIR / "benchmark_imports_for_QandA_notebook")
                return gr.Markdown(
                    f"*`{filename}` not found — place it in `{_hint_dir}/` or `{BASE_DIR}/`*"
                )

            gr.Markdown("### B1 — Whisper Model Sizes")
            _plot("benchmark1.png")

            gr.Markdown("### B2 — WER vs. Audio Bitrate")
            _plot("benchmark2.png")

            gr.Markdown("### B4 — LLM Comparison")
            _plot("benchmark4.png")

            gr.Markdown("### B5 — RAG vs. No-RAG")
            _plot("benchmark5.png")

            gr.Markdown("### B6 — Chunk Size × Whisper Model (MRR Heatmaps)")
            with gr.Row():
                with gr.Column():
                    _plot("benchmark6_short_video_heatmap.png", "Short Video (~5 min)")
                with gr.Column():
                    _plot("benchmark6_medium_video_heatmap.png", "Medium Video (~17 min)")
                with gr.Column():
                    _plot("benchmark6_long_video_heatmap.png", "Long Video (~42 min)")

            gr.Markdown("### B6 — Best MRR per Video × Whisper Model")
            with gr.Row():
                with gr.Column():
                    _plot("benchmark6_short_video_curves.png", "Short Video")
                with gr.Column():
                    _plot("benchmark6_medium_video_curves.png", "Medium Video")
                with gr.Column():
                    _plot("benchmark6_long_video_curves.png", "Long Video")

        with gr.Tab("ℹ️ Credits & Github"):
            gr.Markdown("## 🎥 Multilingual YouTube Analyzer")
            gr.HTML("""
            <div style='text-align: center; padding: 30px; background: linear-gradient(145deg, #1e293b, #0f172a); border-radius: 15px; border: 1px solid #334155; margin-top: 10px;'>
                <div style='margin-bottom: 25px;'>
                    <h3 style='color: #818cf8; text-transform: uppercase; letter-spacing: 2px; font-size: 0.8em; margin-bottom: 10px;'>Assistance</h3>
                    <p style='font-size: 1.5em; color: #f8fafc; font-weight: 600;'>Ashish &bull; Jesus &bull; Nurcan</p>
                </div>
                <div style='height: 1px; background: rgba(255,255,255,0.1); width: 60%; margin: 20px auto;'></div>
                <div style='display: flex; justify-content: center; gap: 30px; margin-top: 20px;'>
                    <div style='flex: 1;'>
                        <p style='color: #f472b6; font-size: 0.7em; text-transform: uppercase; margin-bottom: 5px;'>Special Thanks</p>
                        <p style='color: #e2e8f0; font-weight: 500;'>Al Hussein</p>
                    </div>
                    <div style='flex: 1;'>
                        <p style='color: #38bdf8; font-size: 0.7em; text-transform: uppercase; margin-bottom: 5px;'>AI Intelligence</p>
                        <p style='color: #e2e8f0; font-weight: 500;'>Claude AI</p>
                    </div>
                    <div style='flex: 1;'>
                        <p style='color: #fbbf24; font-size: 0.7em; text-transform: uppercase; margin-bottom: 5px;'>Atmosphere</p>
                        <p style='color: #e2e8f0; font-weight: 500;'>Chill Music Lab</p>
                    </div>
                </div>
            </div>
            <div style='text-align: center; margin-top: 20px;'>
                <a href='https://github.com/DanielIronhack' target='_blank' style='color: #818cf8; text-decoration: none; font-weight: bold;'>
                    📁 Creator Daniel Ironhack
                </a>
            </div>
            """)


    # ═════════════════════════════════════════════════════════════════
    # Event wiring
    # ═════════════════════════════════════════════════════════════════

    def _on_whisper_change(w, lang, agent):
        apply_whisper_choice(w)
        return format_cost_panel(lang, agent, w)

    def _on_model_change(m, lang, w):
        apply_model_choice(m)
        return format_cost_panel(lang, m, w)

    whisper_dropdown.change(
        _on_whisper_change,
        [whisper_dropdown, ui_lang_dropdown, model_dropdown],
        [cost_panel],
    )
    model_dropdown.change(
        _on_model_change,
        [model_dropdown, ui_lang_dropdown, whisper_dropdown],
        [cost_panel],
    )

    # Indexed-video dropdowns fill URL fields
    indexed_dropdown_1.change(
        lambda v: (
            gr.update(value=f"https://www.youtube.com/watch?v={v.split(' — ')[0]}" if v else ""),
            v or "",
        ),
        [indexed_dropdown_1], [url_input, active_video_state],
    )
    indexed_dropdown_2.change(
        lambda v: (
            gr.update(value=f"https://www.youtube.com/watch?v={v.split(' — ')[0]}" if v else ""),
            v or "",
        ),
        [indexed_dropdown_2], [url_input_2, active_video_state],
    )

    # Add buttons — refresh both dropdowns + eval dropdown after ingest
    add_btn.click(
        ui_add_video,
        [url_input, whisper_dropdown, ui_lang_dropdown, model_dropdown],
        [add_status, indexed_dropdown_1, indexed_dropdown_2, video_eval_dropdown, cost_panel],
    ).then(
        lambda d1: d1["value"] if isinstance(d1, dict) and d1.get("value") else "",
        [indexed_dropdown_1], [active_video_state],
    )
    add_btn_2.click(
        ui_add_video_2,
        [url_input_2, whisper_dropdown, ui_lang_dropdown, model_dropdown],
        [add_status_2, indexed_dropdown_1, indexed_dropdown_2, video_eval_dropdown, cost_panel],
    ).then(
        lambda d2: d2["value"] if isinstance(d2, dict) and d2.get("value") else "",
        [indexed_dropdown_2], [active_video_state],
    )

    # Compare button
    compare_btn.click(
        ui_compare_two_videos,
        [indexed_dropdown_1, indexed_dropdown_2, ui_lang_dropdown, model_dropdown],
        [compare_output],
    )

    # Chat
    send_btn.click(
        ui_chat,
        [msg_input, chatbot, session_state, model_dropdown, ui_lang_dropdown, whisper_dropdown, active_video_state],
        [chatbot, session_state, msg_input, cost_panel, passages_output],
    )
    msg_input.submit(
        ui_chat,
        [msg_input, chatbot, session_state, model_dropdown, ui_lang_dropdown, whisper_dropdown, active_video_state],
        [chatbot, session_state, msg_input, cost_panel, passages_output],
    )

    # Audio — fire on .change() (file fully uploaded), not on stop_recording
    # (which can fire before the file is ready). Outputs match the 5 returns
    # of ui_audio_to_chat, including a self-clear of audio_input so the next
    # recording can fire .change() again cleanly.
    audio_input.change(
        ui_audio_to_chat,
        [audio_input, chatbot, session_state, model_dropdown, ui_lang_dropdown,
         whisper_dropdown, active_video_state],
        [chatbot, session_state, audio_input, cost_panel, passages_output],
    )

    def _on_load():
        choices = get_video_dropdown_choices()
        return gr.update(choices=choices, value=choices[0] if choices else None)

    demo.load(_on_load, None, [video_eval_dropdown], js=MIC_JS)

    ui_lang_dropdown.change(
        change_ui_language,
        [ui_lang_dropdown, model_dropdown, whisper_dropdown],
        [
            subtitle_md,
            ui_lang_dropdown,
            settings_header_md,
            whisper_dropdown,
            model_dropdown,
            url_input,
            add_btn,
            url_input_2,
            add_btn_2,
            chat_header_md,
            chatbot,
            msg_input,
            send_btn,
            cost_header_md,
            cost_panel,
            eval_header_md,
            eval_desc_md,
            video_eval_dropdown,
            refresh_eval_btn,
            eval_wer_header_md,
            eval_wer_desc_md,
            wer_btn,
            eval_judge_header_md,
            eval_judge_desc_md,
            judge_btn,
        ],
    )

    refresh_eval_btn.click(
        lambda: gr.update(choices=get_video_dropdown_choices()),
        None,
        [video_eval_dropdown],
    )
    wer_btn.click(ui_compare_whisper_captions, [video_eval_dropdown], [wer_output])
    judge_btn.click(ui_judge_last_answer, [video_eval_dropdown, chatbot, active_video_state], [judge_output])


print("✅ Gradio UI defined")


In [33]:
# Launch UI — share=True creates a public link (valid for 72h)
demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10ae6b0beab5fdc256.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 16. Debug Section

Test individual components without running the full pipeline.

In [34]:
# DEBUG 1: Metadata-only test (fast, no Whisper cost)
def debug_metadata(url):
    try:
        m = get_video_metadata(url)
        print(f"✅ Metadata OK")
        print(f"   ID:       {m.video_id}")
        print(f"   Title:    {m.title}")
        print(f"   Channel:  {m.channel}")
        print(f"   Duration: {m.duration_sec}s")
    except Exception as e:
        print(f"❌ {e}")

# debug_metadata("https://www.youtube.com/watch?v=UF8uR6Z6KLc")
print("debug_metadata(url) — test metadata without Whisper")


debug_metadata(url) — test metadata without Whisper


In [35]:
# DEBUG 2: Audio-Download Test
def debug_download(url):
    try:
        vid = extract_video_id(url)
        path = download_audio(vid)
        size = os.path.getsize(path) / (1024*1024)
        print(f"✅ Audio: {path} ({size:.2f} MB)")
        return path
    except Exception as e:
        print(f"❌ {e}")
        return None

# debug_download("https://www.youtube.com/watch?v=UF8uR6Z6KLc")
print("debug_download(url) — download audio only, no transcription")


debug_download(url) — download audio only, no transcription


In [36]:
# DEBUG 3: Retrieval test (direct, no agent)
def debug_retrieval(query, k=5):
    results = vectorstore.similarity_search(query, k=k)
    print(f"🔍 '{query}' – {len(results)} results\n")
    for i, doc in enumerate(results, 1):
        print(f"{i}. [{doc.metadata.get('title', '?')[:50]}]")
        print(f"   {doc.page_content[:200]}...")
        print()

# debug_retrieval("main topic")
print("debug_retrieval(query) — test retrieval without agent")


debug_retrieval(query) — test retrieval without agent


In [37]:
# DEBUG 4: Direct tool call (no agent)
def debug_tool(tool_name, **kwargs):
    tool_map = {t.name: t for t in AGENT_TOOLS}
    if tool_name not in tool_map:
        print(f"❌ Tool '{tool_name}' not found. Available: {list(tool_map.keys())}")
        return
    result = tool_map[tool_name].invoke(kwargs)
    print(f"Result:\n{result}")

# debug_tool("search_video_content", query="main topic")
# debug_tool("list_indexed_videos")
print("debug_tool(name, **args) — test a tool directly without the agent")


debug_tool(name, **args) — test a tool directly without the agent


In [38]:
# DEBUG 5: ChromaDB inspection
def debug_chromadb():
    try:
        all_docs = vectorstore.get()
        total = len(all_docs.get("ids", []))
        print(f"ChromaDB:")
        print(f"   Collection: {CONFIG['collection_name']}")
        print(f"   Path:       {CONFIG.get('chroma_dir_local', CONFIG.get('chroma_dir', '?'))}")
        print(f"   Chunks:     {total}")
        if total > 0:
            seen = {}
            for meta in all_docs["metadatas"]:
                vid = meta.get("video_id")
                if vid and vid not in seen:
                    seen[vid] = meta.get("title")
            print(f"   Videos:     {len(seen)}")
            for vid, title in seen.items():
                print(f"     - {vid}: {title}")
    except Exception as e:
        print(f"❌ {e}")

debug_chromadb()


ChromaDB:
   Collection: youtube_videos
   Path:       /content/chroma_db
   Chunks:     14
   Videos:     3
     - CYvjC94jDu4: The Only Dating Advice You'll Ever Need!
     - l-rBd0F6d9E: The Art of War’s Greatest Secret (Win Without Fighting)
     - ubjW4aq_MAg: how to *literally* motivate yourself to do anything


## 17. Reset ChromaDB

Run this cell to wipe the vector store and start fresh.


---

## ✅ Done!

- ✅ Audio → Whisper → adaptive chunks → ChromaDB ingestion pipeline
- ✅ 6 Whisper options (API + local: tiny/base/small/medium/large-v3)
- ✅ 6 agent models switchable at runtime (OpenAI + HuggingFace)
- ✅ 7 agent tools (Search, List, Add, Metadata, Summarize, Compare, Key Moments)
- ✅ Two URL inputs with indexed-video dropdowns and compare function
- ✅ Chat: Thinking... indicator, passages + timestamp deep-links inline
- ✅ Microphone input
- ✅ Slim cost panel (current + session totals)
- ✅ Judgement Day: WER check + LLM-as-Judge faithfulness eval
- ✅ Benchmark tab: sweet-spot cards + score/cost-sorted combo table
- ✅ LangSmith EU tracing
- ✅ Auto-indexing of benchmark videos on startup
- ✅ Debug section for each component
- ✅ Multilingual (10 languages)
